# Trading App V2 Option ML Ranker Training

Reuse **trading_app_v2** equity feature engineering + family classifiers, then train an **option meta-stack**:

1. Universe screen + oracle equity labels + option labels (ask→bid, entry filters).
2. **Feature engineering**: fundamental families (`build_fundamental_feature_panel`) **plus** technical families (`build_price_ta_classic_feature_families`: momentum/overlap/candles/math/cycles/performance).
3. **Stage A:** `_train_family_models` on **equity oracle labels only** (same path as Equity MoE) across **all** requested families including technicals.
4. Score family models on option **entry dates** → `long_score` / `short_score` / `net_score` per family.
5. **Stage B:** option meta-ranker on **family scores + option features** (option oracle contracts).
6. Forward eval vs `fixed_near_atm` (`entry_date >= 2025-01-01`).


In [1]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
import json
import sys
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 260)

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "app").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def prefer_local_repo(package_name: str, repo_dir: Path) -> None:
    repo_dir = repo_dir.resolve()
    if not repo_dir.exists():
        return
    repo_text = str(repo_dir)
    sys.path[:] = [
        entry
        for entry in sys.path
        if str(Path(entry or ".").expanduser().resolve()) != repo_text
    ]
    sys.path.insert(0, repo_text)
    loaded = sys.modules.get(package_name)
    loaded_file = Path(str(getattr(loaded, "__file__", ""))).resolve() if loaded is not None else None
    if loaded_file is not None and repo_dir not in loaded_file.parents:
        for module_name in [name for name in sys.modules if name == package_name or name.startswith(f"{package_name}.")]:
            del sys.modules[module_name]


WORKSPACE_ROOT = REPO_ROOT.parent
prefer_local_repo("quant_orchestrator", WORKSPACE_ROOT / "quant-orchestrator")
prefer_local_repo("quant_warehouse", WORKSPACE_ROOT / "quant-warehouse")
prefer_local_repo("fmpsdk", WORKSPACE_ROOT / "fmpsdk")

from app.quant_warehouse_storage import ensure_quant_warehouse_storage
from app.trading_app_v2_runtime import DEFAULT_STRATEGY_SOURCES, default_paths

paths = default_paths(REPO_ROOT)
paths.artifact_root.mkdir(parents=True, exist_ok=True)
paths.option_artifact_dir.mkdir(parents=True, exist_ok=True)

print(
    {
        "repo_root": str(REPO_ROOT),
        "artifact_root": str(paths.artifact_root),
        "option_artifact_dir": str(paths.option_artifact_dir),
    }
)

{'repo_root': '/home/jlee153232/PycharmProjects/optimal_trader', 'artifact_root': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2', 'option_artifact_dir': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2/option_family_ranker'}


## Environment

Assumes `quant-warehouse` and `quant-orchestrator` are already installed. This notebook only ensures warehouse storage paths are visible; it does not download option chains or stop background ThetaData jobs.

In [2]:
storage = ensure_quant_warehouse_storage()
display(storage.as_dict())


def optional_date(value):
    if value is None:
        return None
    text = str(value).strip()
    if not text or text.lower() in {"none", "null", "nat"}:
        return None
    return text

{'home': '/home/jlee153232/.quant-warehouse',
 'arctic_uri': 'lmdb:///home/jlee153232/.quant-warehouse/arctic',
 'catalog_path': '/home/jlee153232/.quant-warehouse/catalog.sqlite',
 'provider_arctic_uris': {'fmp': 'lmdb:///home/jlee153232/.quant-warehouse/arctic/providers/fmp',
  'thetadata': 'lmdb:///home/jlee153232/.quant-warehouse/arctic/providers/thetadata'}}

## Run Configuration

Start config: **$1T** min market cap and oracle **YE k = 1..12**. Option chain reads are **cache-only** (`download_missing=False`) so this notebook does not refresh ThetaData or touch an in-flight refresh job.

In [3]:
RUN_UNIVERSE_SCREEN = True
RUN_ORACLE_TRADE_LABELS = True
RUN_OPTION_COVERAGE = True
RUN_OPTION_LABEL_PANEL = True
RUN_EQUITY_FAMILY_MODELS = True  # trading_app_v2 _train_family_models on equity oracle labels
RUN_OPTION_META_STACK = True  # option ranker on family scores + option features
RUN_OPTION_RANKER_TRAINING = True  # run the equity+option meta-stack cell

DATA_START = "1900-01-01"
DATA_END = ""  # empty = latest available warehouse data
MIN_MARKET_CAP = 100_000_000_000  # $100B
ORACLE_YE_K = (1, 2, 3)  # production target set
ORACLE_MIN_PROFIT_PCT = 0.01

# Cache-only ThetaData option labels
OPTION_MAX_DTE = None  # retain early expirations for unified behavior labels
OPTION_TARGET_DTE = 90
MAX_ORACLE_TRADES_FOR_OPTIONS = 0
OPTION_PANEL_PROGRESS_EVERY = 50

OPTION_PANEL_PATH = paths.artifact_root / "option_candidate_panel.parquet"
ORACLE_TRADES_PATH = paths.option_artifact_dir / "oracle_trade_windows.parquet"
ORACLE_TRADES_UNIQUE_PATH = paths.option_artifact_dir / "oracle_trade_windows_unique.parquet"
ORACLE_LABEL_DIAG_PATH = paths.option_artifact_dir / "oracle_label_diagnostics.csv"
OPTION_LABEL_SUMMARY_PATH = paths.option_artifact_dir / "option_label_build_summary.json"
ORACLE_TRADES_LABELED_PATH = paths.option_artifact_dir / "oracle_trade_windows_with_options.parquet"
ORACLE_TRADES_SKIPPED_PATH = paths.option_artifact_dir / "oracle_trade_windows_skipped_missing_options.parquet"
OPTION_COVERAGE_PATH = paths.option_artifact_dir / "option_chain_coverage_by_symbol.csv"
ORACLE_TRADES_IN_COVERAGE_PATH = paths.option_artifact_dir / "oracle_trade_windows_in_option_coverage.parquet"
ORACLE_TRADES_BEFORE_COVERAGE_PATH = paths.option_artifact_dir / "oracle_trade_windows_before_option_coverage.parquet"

# Temporal split + meta knobs
TRAIN_ON_ALL_DATA = False
OPTION_TRAIN_END = "2024-12-31"  # equity train + option meta train by entry_date
OPTION_EVAL_START = "2025-01-01"  # forward
TARGET_COL = "rank_y"
TRAIN_TOP_K_BY_RETURN = 128  # legacy name: diverse bounded candidates; preserves unified rank_y
N_ESTIMATORS = 400
MAX_TRADES = 0
RANDOM_SEED = 20260704
DISABLE_PAIRWISE_RANKER = True
# Feature families: trading_app_v2 defaults (fundamentals/macro) + optional technicals
from quant_warehouse.platforms.data_providers.fmp.feature_engineering import TA_CLASSIC_FAMILY_PREFIXES

TECHNICAL_STRATEGY_SOURCES = tuple(f"fmp.{family}" for family in TA_CLASSIC_FAMILY_PREFIXES)
INCLUDE_TECHNICAL_FEATURE_FAMILIES = True  # momentum / overlap / candles / math / cycles / performance
# True → keep every family present after FE (fundamentals + technicals if built)
ALL_FEATURE_FAMILIES = True
REQUIRE_ALL_REQUESTED_FEATURE_FAMILIES = True

_base_families = tuple(DEFAULT_STRATEGY_SOURCES)
if INCLUDE_TECHNICAL_FEATURE_FAMILIES:
    FEATURE_FAMILIES = tuple(dict.fromkeys([*_base_families, *TECHNICAL_STRATEGY_SOURCES]))
else:
    FEATURE_FAMILIES = _base_families

display(
    {
        "min_market_cap": MIN_MARKET_CAP,
        "oracle_ye_k": list(ORACLE_YE_K),
        "option_train_end": OPTION_TRAIN_END,
        "option_eval_start": OPTION_EVAL_START,
        "train_top_k_by_return": TRAIN_TOP_K_BY_RETURN,
        "run_equity_family_models": RUN_EQUITY_FAMILY_MODELS,
        "run_option_meta_stack": RUN_OPTION_META_STACK,
        "all_feature_families": ALL_FEATURE_FAMILIES,
        "include_technical_feature_families": INCLUDE_TECHNICAL_FEATURE_FAMILIES,
        "feature_families": len(FEATURE_FAMILIES),
        "technical_families": list(TECHNICAL_STRATEGY_SOURCES) if INCLUDE_TECHNICAL_FEATURE_FAMILIES else [],
        "n_estimators": N_ESTIMATORS,
    }
)


{'min_market_cap': 100000000000,
 'oracle_ye_k': [1, 2, 3],
 'option_train_end': '2024-12-31',
 'option_eval_start': '2025-01-01',
 'train_top_k_by_return': 128,
 'run_equity_family_models': True,
 'run_option_meta_stack': True,
 'all_feature_families': True,
 'include_technical_feature_families': True,
 'feature_families': 28,
 'technical_families': ['fmp.technical_candles',
  'fmp.technical_cycles',
  'fmp.technical_math',
  'fmp.technical_momentum',
  'fmp.technical_overlap',
  'fmp.technical_performance'],
 'n_estimators': 400}

## Universe Screen ($1T)

Screen the FMP equity universe at the configured min market cap. This cell does **not** refresh FMP historical data; it only lists eligible symbols from the warehouse/OpenBB screen path.

In [4]:
from quant_warehouse.research_tools.feature_family_eval import FamilyEvaluationConfig, screen_fmp_equity_universe
from quant_warehouse.warehouse.api import Warehouse

warehouse = Warehouse()
screen_config = FamilyEvaluationConfig(
    provider="fmp",
    market_cap_min=MIN_MARKET_CAP,
    start_date=DATA_START,
    end_date=optional_date(DATA_END),
)

if RUN_UNIVERSE_SCREEN:
    screened_equity_symbols, raw_fmp_universe, universe_eligibility, universe_source = screen_fmp_equity_universe(
        screen_config,
        warehouse=warehouse,
    )
    screened_equity_symbols = tuple(str(s).strip().upper() for s in screened_equity_symbols if str(s).strip())
    print(
        f"Universe source={universe_source}; symbols={len(screened_equity_symbols):,}; "
        f"min_market_cap={MIN_MARKET_CAP:,}"
    )
    display(pd.DataFrame({"symbol": list(screened_equity_symbols)}))
    display(universe_eligibility.head(50))
else:
    raise RuntimeError("Set RUN_UNIVERSE_SCREEN=True before building oracle labels.")


Universe source=openbb:fmp; symbols=118; min_market_cap=100,000,000,000


,symbol
0,AAPL
1,ABBV
2,ABT
3,ADI
4,ADP
...,...
113,WDC
114,WELL
115,WFC
116,WMT


,symbol,eligible,reason,screen_market_cap
0,NVDA,True,ok,4929700130000
1,AAPL,True,ok,4660444932360
2,GOOG,True,ok,4266992790966
3,GOOGL,True,ok,4263562726985
4,MSFT,True,ok,2904441845700
5,AMZN,True,ok,2660338401000
6,AVGO,True,ok,1827148599000
7,SPCX,True,ok,1819375917600
8,META,True,ok,1667058815244
9,TSLA,True,ok,1482608027200


## Oracle Trade Labels (YE 1..12)

Build side-specific oracle equity trade windows from warehouse prices:
- frequency `YE`
- `k = 1..12`
- long entry/exit on high/low, short entry/exit on low/high (same convention as `trading_app_v2`)

**Important:** many k values emit the **same** `(symbol, side, entry_date, exit_date)` window. For options we only need each unique window once, so this cell also builds a deduped table:
- `oracle_trade_windows` — full multi-k rows (for diagnostics)
- `oracle_trade_windows_unique` — one row per `(symbol, side, entry_date, exit_date)` used for coverage filtering and option labels


In [5]:
from quant_warehouse.platforms.data_providers.fmp.target_engineering import LabelBuildSpec, build_trade_results
from quant_warehouse.research_tools import BinaryTargetConfig
from quant_orchestrator.research_tools.ml_trading_experiment import _build_oracle_trade_label_rows_sparse

if "screened_equity_symbols" not in globals() or not screened_equity_symbols:
    raise RuntimeError("Run the Universe Screen cell first.")

if RUN_ORACLE_TRADE_LABELS:
    target_config = BinaryTargetConfig(
        provider="fmp",
        start_date=DATA_START,
        end_date=optional_date(DATA_END),
        oracle_trade_k_by_frequency={"YE": tuple(ORACLE_YE_K)},
        oracle_trade_min_profit_pct=ORACLE_MIN_PROFIT_PCT,
    )

    oracle_label_rows, oracle_label_diagnostics, oracle_seconds, _sparse_oracle_trade_windows_unique = (
        _build_oracle_trade_label_rows_sparse(
            screened_equity_symbols,
            target_config,
            warehouse=warehouse,
        )
    )

    oracle_price_frames: dict[str, pd.DataFrame] = {}
    for raw_symbol in screened_equity_symbols:
        symbol = str(raw_symbol).strip().upper()
        if not symbol:
            continue
        prices = warehouse.read_prices(
            symbol,
            provider=target_config.provider,
            start=target_config.start_date,
            end=target_config.end_date,
        )
        if prices is not None and not prices.empty:
            oracle_price_frames[symbol] = prices

    oracle_trade_spec = LabelBuildSpec(
        k_params={frequency: list(ks) for frequency, ks in target_config.oracle_trade_k_by_frequency.items()},
        min_profit_pct=target_config.oracle_trade_min_profit_pct,
        buy_execution=target_config.oracle_trade_long_entry_price_col,
        sell_execution=target_config.oracle_trade_long_exit_price_col,
        short_execution=target_config.oracle_trade_short_entry_price_col,
        cover_execution=target_config.oracle_trade_short_exit_price_col,
        start_date=target_config.start_date,
        end_date=target_config.end_date,
    )
    oracle_trade_result = build_trade_results(
        list(oracle_price_frames),
        spec=oracle_trade_spec,
        price_frames=oracle_price_frames,
    )
    oracle_trade_windows = pd.DataFrame(oracle_trade_result.completed_trades)
    if oracle_trade_windows.empty:
        raise RuntimeError("No oracle trade windows were produced for the screened universe.")

    oracle_trade_windows["symbol"] = oracle_trade_windows["symbol"].astype(str).str.upper()
    oracle_trade_windows["side"] = oracle_trade_windows["side"].astype(str).str.lower().str.strip()
    oracle_trade_windows["entry_date"] = pd.to_datetime(oracle_trade_windows["entry_date"], errors="coerce").dt.normalize()
    oracle_trade_windows["exit_date"] = pd.to_datetime(oracle_trade_windows["exit_date"], errors="coerce").dt.normalize()
    oracle_trade_windows = oracle_trade_windows.dropna(subset=["symbol", "side", "entry_date", "exit_date"])
    oracle_trade_windows = oracle_trade_windows.loc[
        oracle_trade_windows["side"].isin({"long", "short"})
        & oracle_trade_windows["exit_date"].gt(oracle_trade_windows["entry_date"])
    ].copy()

    if "trade_id" not in oracle_trade_windows.columns:
        oracle_trade_windows["trade_id"] = (
            oracle_trade_windows["symbol"].astype(str)
            + "|"
            + oracle_trade_windows["side"].astype(str)
            + "|"
            + oracle_trade_windows["entry_date"].dt.strftime("%Y%m%d")
            + "|"
            + oracle_trade_windows["exit_date"].dt.strftime("%Y%m%d")
            + "|k"
            + oracle_trade_windows["k"].astype(str)
        )

    oracle_trade_windows = oracle_trade_windows.sort_values(
        ["entry_date", "symbol", "side", "k", "trade_id"],
        ascending=[False, True, True, True, True],
        kind="stable",
    ).reset_index(drop=True)

    # YE k=1..12 often reuses the same entry/exit window across k. Collapse to unique windows
    # for option labeling: one row per (symbol, side, entry_date, exit_date).
    _base = oracle_trade_windows.copy()
    _base["k"] = pd.to_numeric(_base["k"], errors="coerce") if "k" in _base.columns else pd.Series(pd.NA, index=_base.index)
    agg_map = {
        "k_min": ("k", "min"),
        "k_max": ("k", "max"),
        "k_count": ("k", "nunique"),
        "multi_k_rows": ("trade_id", "size"),
    }
    for col, fn in (
        ("freq", "first"),
        ("entry_px", "first"),
        ("exit_px", "first"),
        ("ret_dec", "first"),
        ("ret_pct", "first"),
        ("hold_days", "first"),
    ):
        if col in _base.columns:
            agg_map[col] = (col, fn)
    oracle_trade_windows_unique = _base.groupby(
        ["symbol", "side", "entry_date", "exit_date"], as_index=False
    ).agg(**agg_map)
    k_lists = (
        _base.groupby(["symbol", "side", "entry_date", "exit_date"])["k"]
        .agg(lambda s: ",".join(str(int(v)) for v in sorted(pd.Series(s).dropna().unique())))
        .reset_index(name="k_values")
    )
    oracle_trade_windows_unique = oracle_trade_windows_unique.merge(
        k_lists, on=["symbol", "side", "entry_date", "exit_date"], how="left"
    )
    oracle_trade_windows_unique["k"] = oracle_trade_windows_unique["k_min"]
    oracle_trade_windows_unique["trade_id"] = (
        oracle_trade_windows_unique["symbol"].astype(str)
        + "|"
        + oracle_trade_windows_unique["side"].astype(str)
        + "|"
        + oracle_trade_windows_unique["entry_date"].dt.strftime("%Y%m%d")
        + "|"
        + oracle_trade_windows_unique["exit_date"].dt.strftime("%Y%m%d")
    )
    oracle_trade_windows_unique = oracle_trade_windows_unique.sort_values(
        ["entry_date", "symbol", "side", "trade_id"],
        ascending=[False, True, True, True],
        kind="stable",
    ).reset_index(drop=True)

    ORACLE_TRADES_PATH.parent.mkdir(parents=True, exist_ok=True)
    oracle_trade_windows.to_parquet(ORACLE_TRADES_PATH, index=False)
    oracle_trade_windows_unique.to_parquet(ORACLE_TRADES_UNIQUE_PATH, index=False)
    if isinstance(oracle_label_diagnostics, pd.DataFrame) and not oracle_label_diagnostics.empty:
        oracle_label_diagnostics.to_csv(ORACLE_LABEL_DIAG_PATH, index=False)

    dup_share = 1.0 - (len(oracle_trade_windows_unique) / max(len(oracle_trade_windows), 1))
    print(
        {
            "oracle_label_rows": len(oracle_label_rows),
            "oracle_classes": sorted(oracle_label_rows["collapsed_label"].dropna().unique())
            if isinstance(oracle_label_rows, pd.DataFrame) and not oracle_label_rows.empty
            else [],
            "oracle_seconds": round(float(oracle_seconds), 3),
            "oracle_trade_windows_multi_k": len(oracle_trade_windows),
            "oracle_trade_windows_unique": len(oracle_trade_windows_unique),
            "duplicate_window_share": round(float(dup_share), 4),
            "symbols": int(oracle_trade_windows_unique["symbol"].nunique()),
            "long_unique": int(oracle_trade_windows_unique["side"].eq("long").sum()),
            "short_unique": int(oracle_trade_windows_unique["side"].eq("short").sum()),
            "saved_multi_k": str(ORACLE_TRADES_PATH),
            "saved_unique": str(ORACLE_TRADES_UNIQUE_PATH),
        }
    )
    display(oracle_trade_windows.groupby(["side", "k"], dropna=False).size().rename("trades").reset_index().head(40))
    display(
        oracle_trade_windows_unique.groupby("k_count", dropna=False)
        .size()
        .rename("unique_windows")
        .reset_index()
        .sort_values("k_count")
    )
    display(oracle_trade_windows_unique.head(20))
else:
    if not ORACLE_TRADES_PATH.exists():
        raise FileNotFoundError(f"Missing oracle trades at {ORACLE_TRADES_PATH}")
    oracle_trade_windows = pd.read_parquet(ORACLE_TRADES_PATH)
    if ORACLE_TRADES_UNIQUE_PATH.exists():
        oracle_trade_windows_unique = pd.read_parquet(ORACLE_TRADES_UNIQUE_PATH)
    else:
        _base = oracle_trade_windows.copy()
        _base["symbol"] = _base["symbol"].astype(str).str.upper()
        _base["side"] = _base["side"].astype(str).str.lower()
        _base["entry_date"] = pd.to_datetime(_base["entry_date"], errors="coerce").dt.normalize()
        _base["exit_date"] = pd.to_datetime(_base["exit_date"], errors="coerce").dt.normalize()
        _base["k"] = pd.to_numeric(_base["k"], errors="coerce") if "k" in _base.columns else pd.Series(pd.NA, index=_base.index)
        oracle_trade_windows_unique = (
            _base.groupby(["symbol", "side", "entry_date", "exit_date"], as_index=False)
            .agg(k_min=("k", "min"), k_max=("k", "max"), k_count=("k", "nunique"), multi_k_rows=("trade_id", "size"))
        )
        oracle_trade_windows_unique["k"] = oracle_trade_windows_unique["k_min"]
        oracle_trade_windows_unique["trade_id"] = (
            oracle_trade_windows_unique["symbol"].astype(str)
            + "|"
            + oracle_trade_windows_unique["side"].astype(str)
            + "|"
            + oracle_trade_windows_unique["entry_date"].dt.strftime("%Y%m%d")
            + "|"
            + oracle_trade_windows_unique["exit_date"].dt.strftime("%Y%m%d")
        )
        oracle_trade_windows_unique.to_parquet(ORACLE_TRADES_UNIQUE_PATH, index=False)
    print(
        f"Loaded oracle trades multi_k={len(oracle_trade_windows):,} "
        f"unique={len(oracle_trade_windows_unique):,}"
    )


{'oracle_label_rows': 28833, 'oracle_classes': ['oracle_long', 'oracle_short'], 'oracle_seconds': 5.911, 'oracle_trade_windows_multi_k': 57697, 'oracle_trade_windows_unique': 36106, 'duplicate_window_share': 0.3742, 'symbols': 118, 'long_unique': 18967, 'short_unique': 17139, 'saved_multi_k': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2/option_family_ranker/oracle_trade_windows.parquet', 'saved_unique': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2/option_family_ranker/oracle_trade_windows_unique.parquet'}


,side,k,trades
0,long,1,4815
1,long,2,9621
2,long,3,14420
3,short,1,4812
4,short,2,9616
5,short,3,14413


,k_count,unique_windows
0,1,19126
1,2,12369
2,3,4611


,symbol,side,entry_date,exit_date,k_min,k_max,k_count,multi_k_rows,freq,entry_px,exit_px,ret_dec,ret_pct,hold_days,k_values,k,trade_id
0,MRVL,long,2026-07-08,2026-07-09,3,3,1,1,YE,236.7300,241.0400,0.018206,1.82%,1,3,3,MRVL|long|20260708|20260709
1,AVGO,long,2026-07-07,2026-07-10,3,3,1,1,YE,372.9600,395.7000,0.060972,6.10%,3,3,3,AVGO|long|20260707|20260710
2,BA,short,2026-07-07,2026-07-10,3,3,1,1,YE,230.4600,223.8400,0.028725,2.87%,3,3,3,BA|short|20260707|20260710
3,MCD,short,2026-07-07,2026-07-09,3,3,1,1,YE,281.3800,277.5800,0.013505,1.35%,2,3,3,MCD|short|20260707|20260709
4,PEP,short,2026-07-07,2026-07-10,2,3,2,2,YE,144.5400,137.9600,0.045524,4.55%,3,"2,3",2,PEP|short|20260707|20260710
5,SNDK,long,2026-07-07,2026-07-09,2,3,2,2,YE,"1,638.8800","1,801.0000",0.098921,9.89%,2,"2,3",2,SNDK|long|20260707|20260709
6,CVX,long,2026-07-01,2026-07-08,3,3,1,1,YE,167.4700,174.9500,0.044665,4.47%,7,3,3,CVX|long|20260701|20260708
7,DE,short,2026-07-01,2026-07-10,3,3,1,1,YE,619.3000,599.4400,0.032068,3.21%,9,3,3,DE|short|20260701|20260710
8,HD,short,2026-07-01,2026-07-08,3,3,1,1,YE,350.8400,337.9900,0.036626,3.66%,7,3,3,HD|short|20260701|20260708
9,AMAT,short,2026-06-30,2026-07-07,1,3,3,3,YE,696.0000,558.2800,0.197874,19.79%,7,"1,2,3",1,AMAT|short|20260630|20260707


## Feature Engineering

Same path as `trading_app_v2` for fundamentals, extended with **technical feature families**:

1. `build_fundamental_feature_panel` — FMP/FinanceToolkit/macro families (`DEFAULT_STRATEGY_SOURCES`)
2. `build_price_ta_classic_feature_families` — per-symbol price technicals split into:
   - `fmp.technical_momentum`
   - `fmp.technical_overlap`
   - `fmp.technical_candles`
   - `fmp.technical_math`
   - `fmp.technical_cycles`
   - `fmp.technical_performance`
3. Quality cap + family filter (`ALL_FEATURE_FAMILIES` keeps everything built; otherwise uses `FEATURE_FAMILIES`)

Stage A trains one equity classifier per `(source, family)` including the technical families when present.


In [6]:
from quant_warehouse.research_tools import (
    FamilyEvaluationConfig,
    FeatureSpec,
    build_fundamental_feature_panel,
    cap_features_by_quality,
)
from quant_warehouse.platforms.data_providers.fmp.feature_engineering import (
    TA_CLASSIC_FAMILY_PREFIXES,
    build_price_ta_classic_feature_families,
)
from quant_warehouse.warehouse.api import Warehouse

if "screened_equity_symbols" not in globals():
    raise RuntimeError("Run the Universe Screen cell before feature engineering.")


def _slice_prices(frame: pd.DataFrame, start: str | None, end: str | None) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    out = frame.copy()
    out.index = pd.to_datetime(out.index, errors="coerce")
    out = out.loc[out.index.notna()].sort_index()
    if start is not None:
        out = out.loc[out.index >= pd.Timestamp(start)]
    if end is not None:
        out = out.loc[out.index <= pd.Timestamp(end)]
    return out


def build_technical_feature_panel(
    symbols: list[str] | tuple[str, ...],
    config: FamilyEvaluationConfig,
    *,
    warehouse: Warehouse,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, float]]:
    """Build ta-classic technical families (source=fmp) aligned to daily prices."""
    started = perf_counter()
    frames: list[pd.DataFrame] = []
    specs: list[FeatureSpec] = []
    diagnostics: list[dict] = []
    for symbol in symbols:
        sym = str(symbol).strip().upper()
        t0 = perf_counter()
        try:
            prices = warehouse.read_prices(sym, provider=config.provider)
        except Exception as exc:
            diagnostics.append(
                {
                    "symbol": sym,
                    "status": f"read_prices_error:{type(exc).__name__}",
                    "seconds": perf_counter() - t0,
                }
            )
            continue
        prices = _slice_prices(prices, config.start_date, config.end_date)
        if prices.empty:
            diagnostics.append({"symbol": sym, "status": "empty_prices", "seconds": perf_counter() - t0})
            continue
        try:
            built_by_family = build_price_ta_classic_feature_families(sym, prices)
        except Exception as exc:
            diagnostics.append(
                {
                    "symbol": sym,
                    "status": f"ta_classic_error:{type(exc).__name__}",
                    "seconds": perf_counter() - t0,
                    "error": str(exc)[:200],
                }
            )
            continue

        feature_frames: dict[str, pd.Series] = {}
        n_features = 0
        for family_name, built in built_by_family.items():
            if built is None or not getattr(built, "feature_cols", None):
                continue
            df = built.df
            if df is None or df.empty:
                continue
            # BuiltFeatureSet is MultiIndex (date, symbol) or date index
            work = df.reset_index()
            if "date" not in work.columns:
                # first level may be unnamed date
                for c in work.columns:
                    if c not in {"symbol"} and pd.api.types.is_datetime64_any_dtype(work[c]):
                        work = work.rename(columns={c: "date"})
                        break
            if "date" not in work.columns:
                continue
            work["date"] = pd.to_datetime(work["date"], errors="coerce").dt.normalize()
            work = work.dropna(subset=["date"])
            for col in built.feature_cols:
                if col not in work.columns:
                    continue
                feature_frames[col] = pd.Series(
                    pd.to_numeric(work[col], errors="coerce").to_numpy(),
                    index=pd.DatetimeIndex(work["date"]),
                    name=col,
                )
                specs.append(
                    FeatureSpec(
                        feature=col,
                        family=str(family_name),
                        source="fmp",
                        source_column=col,
                        expected_direction="higher_is_better",
                    )
                )
                n_features += 1
        if not feature_frames:
            diagnostics.append(
                {
                    "symbol": sym,
                    "status": "no_technical_features",
                    "seconds": perf_counter() - t0,
                }
            )
            continue
        tech = pd.DataFrame(feature_frames)
        tech["date"] = tech.index
        tech["symbol"] = sym
        tech = tech.reset_index(drop=True)
        frames.append(tech)
        diagnostics.append(
            {
                "symbol": sym,
                "status": "ok",
                "rows": int(len(tech)),
                "features": int(n_features),
                "families": int(len({s.family for s in specs if s.feature in tech.columns})),
                "seconds": perf_counter() - t0,
            }
        )

    timings = {"technical_panel_build_seconds": perf_counter() - started}
    if not frames:
        empty_panel = pd.DataFrame(columns=["date", "symbol"])
        empty_meta = pd.DataFrame(columns=["feature", "family", "source", "source_column", "expected_direction"])
        return empty_panel, empty_meta, pd.DataFrame(diagnostics), timings

    panel = pd.concat(frames, ignore_index=True, sort=False)
    panel["symbol"] = panel["symbol"].astype(str).str.upper()
    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize()
    metadata = (
        pd.DataFrame([spec.__dict__ for spec in specs])
        .drop_duplicates(subset=["feature"])
        .sort_values(["family", "feature"])
        .reset_index(drop=True)
    )
    return panel, metadata, pd.DataFrame(diagnostics), timings


engineering_warehouse = Warehouse()
feature_config = FamilyEvaluationConfig(
    provider="fmp",
    market_cap_min=MIN_MARKET_CAP,
    start_date=DATA_START,
    end_date=optional_date(DATA_END),
)

# 1) Fundamentals / macro (trading_app_v2 path)
raw_feature_panel, raw_feature_metadata, feature_diagnostics, feature_timings = build_fundamental_feature_panel(
    screened_equity_symbols,
    feature_config,
    warehouse=engineering_warehouse,
)

# 2) Technical families (optional)
technical_timings: dict[str, float] = {}
technical_diagnostics = pd.DataFrame()
if INCLUDE_TECHNICAL_FEATURE_FAMILIES:
    print(
        f"[features] building technical families for {len(screened_equity_symbols)} symbols "
        f"({', '.join(TA_CLASSIC_FAMILY_PREFIXES)}) ...",
        flush=True,
    )
    tech_panel, tech_metadata, technical_diagnostics, technical_timings = build_technical_feature_panel(
        screened_equity_symbols,
        feature_config,
        warehouse=engineering_warehouse,
    )
    print(
        {
            "technical_rows": len(tech_panel),
            "technical_features": 0 if tech_metadata.empty else int(tech_metadata["feature"].nunique()),
            "technical_families": 0 if tech_metadata.empty else int(tech_metadata["family"].nunique()),
            **technical_timings,
        },
        flush=True,
    )
    if not tech_panel.empty and not tech_metadata.empty:
        # Outer-merge so technical-only dates/symbols are kept if fundamentals missing a day
        key_cols = ["symbol", "date"]
        fund_cols = [c for c in raw_feature_panel.columns if c not in key_cols]
        tech_cols = [c for c in tech_panel.columns if c not in key_cols]
        overlap = sorted(set(fund_cols).intersection(tech_cols))
        if overlap:
            # Prefer fundamental columns on name clash (should be rare)
            tech_panel = tech_panel.drop(columns=overlap)
            tech_cols = [c for c in tech_cols if c not in overlap]
        raw_feature_panel = raw_feature_panel.merge(
            tech_panel[key_cols + tech_cols],
            on=key_cols,
            how="outer",
            sort=False,
        )
        raw_feature_metadata = (
            pd.concat([raw_feature_metadata, tech_metadata], ignore_index=True, sort=False)
            .drop_duplicates(subset=["feature"], keep="first")
            .sort_values(["source", "family", "feature"])
            .reset_index(drop=True)
        )
        feature_timings = {**feature_timings, **technical_timings}
    else:
        print("[features] technical panel empty — continuing with fundamentals only", flush=True)

selected_features, selected_feature_metadata, feature_quality = cap_features_by_quality(
    raw_feature_panel,
    raw_feature_metadata,
    max_features=None,
)

# Family filter
raw_metadata = raw_feature_metadata.copy()
raw_metadata["strategy_source"] = raw_metadata["source"].astype(str) + "." + raw_metadata["family"].astype(str)
raw_available_sources = set(raw_metadata["strategy_source"].astype(str))

if ALL_FEATURE_FAMILIES:
    wanted_sources = set(raw_available_sources)
    # Still ensure requested defaults (+ technicals if enabled) are present when required
    required_sources = {str(s).strip() for s in FEATURE_FAMILIES}
else:
    wanted_sources = {str(source).strip() for source in FEATURE_FAMILIES}
    required_sources = set(wanted_sources)

missing_raw_sources = sorted(required_sources.difference(raw_available_sources))
metadata = selected_feature_metadata.copy()
metadata["strategy_source"] = metadata["source"].astype(str) + "." + metadata["family"].astype(str)
selected_feature_metadata = (
    metadata.loc[metadata["strategy_source"].isin(wanted_sources)]
    .drop(columns=["strategy_source"])
    .reset_index(drop=True)
)
selected_available_sources = set(
    selected_feature_metadata["source"].astype(str) + "." + selected_feature_metadata["family"].astype(str)
)
missing_selected_sources = sorted(required_sources.difference(selected_available_sources))
if REQUIRE_ALL_REQUESTED_FEATURE_FAMILIES and (missing_raw_sources or missing_selected_sources):
    raise RuntimeError(
        "Requested feature families are missing. "
        f"missing_raw={missing_raw_sources}; missing_selected={missing_selected_sources}"
    )
selected_features = [
    feature
    for feature in selected_features
    if feature in set(selected_feature_metadata["feature"].astype(str))
]
# Keep only columns that exist on the panel
selected_features = [f for f in selected_features if f in raw_feature_panel.columns]
feature_panel = raw_feature_panel[["symbol", "date", *selected_features]].copy()
feature_panel["symbol"] = feature_panel["symbol"].astype(str).str.upper()
feature_panel["date"] = pd.to_datetime(feature_panel["date"], errors="coerce").dt.normalize()

family_counts = (
    selected_feature_metadata.groupby(["source", "family"], as_index=False)
    .agg(feature_count=("feature", "nunique"))
    .sort_values(["source", "family"])
)
family_counts["strategy_source"] = family_counts["source"].astype(str) + "." + family_counts["family"].astype(str)
family_counts["is_technical"] = family_counts["family"].astype(str).str.startswith("technical_")

print(
    {
        "symbols": len(screened_equity_symbols),
        "raw_feature_panel_rows": len(raw_feature_panel),
        "selected_feature_columns": len(selected_features),
        "selected_feature_families": int(selected_feature_metadata["family"].nunique()),
        "requested_feature_families": len(required_sources),
        "all_feature_families": ALL_FEATURE_FAMILIES,
        "include_technical": INCLUDE_TECHNICAL_FEATURE_FAMILIES,
        "technical_families_selected": int(family_counts["is_technical"].sum()),
        "missing_requested_raw_families": missing_raw_sources,
        "missing_requested_selected_families": missing_selected_sources,
        **feature_timings,
    }
)
display(feature_diagnostics.head(20))
if INCLUDE_TECHNICAL_FEATURE_FAMILIES and not technical_diagnostics.empty:
    print("Technical build diagnostics (head):")
    display(technical_diagnostics.head(20))
print("Selected families (fundamentals + technicals):")
display(family_counts)


/home/jlee153232/PycharmProjects/quant-warehouse/quant_warehouse/research_tools/feature_family_eval.py:756: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel[feature] = series
/home/jlee153232/PycharmProjects/quant-warehouse/quant_warehouse/research_tools/feature_family_eval.py:756: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel[feature] = series
/home/jlee153232/PycharmProjects/quant-warehouse/quant_warehouse/research_tools/feature_family_eval.py:756: PerformanceWarning: DataFrame is highly fragmented.  This is usually t

[features] building technical families for 118 symbols (technical_candles, technical_cycles, technical_math, technical_momentum, technical_overlap, technical_performance) ...
{'technical_rows': 1186974, 'technical_features': 187, 'technical_families': 6, 'technical_panel_build_seconds': 209.747962763533}
{'symbols': 118, 'raw_feature_panel_rows': 1186974, 'selected_feature_columns': 667, 'selected_feature_families': 28, 'requested_feature_families': 28, 'all_feature_families': True, 'include_technical': True, 'technical_families_selected': 6, 'missing_requested_raw_families': [], 'missing_requested_selected_families': [], 'raw_panel_build_seconds': 25.89969018381089, 'technical_panel_build_seconds': 209.747962763533}


,symbol,status,rows,features,seconds
0,AAPL,ok,11485,376,0.136990
1,ABBV,ok,3415,376,0.105388
2,ABT,ok,14724,376,0.119324
3,ADI,ok,13639,376,0.142016
4,ADP,ok,14727,376,0.134619
5,AMAT,ok,13459,376,0.135736
6,AMD,ok,11673,376,0.104070
7,AMGN,ok,10850,376,0.127994
8,AMZN,ok,5882,376,0.110359
9,ANET,ok,3041,376,0.094986


Technical build diagnostics (head):


,symbol,status,rows,features,families,seconds
0,AAPL,ok,11485,187,6,2.275019
1,ABBV,ok,3415,187,6,0.676572
2,ABT,ok,14724,187,6,2.512643
3,ADI,ok,13639,187,6,2.370576
4,ADP,ok,14727,187,6,2.522094
5,AMAT,ok,13459,187,6,2.318858
6,AMD,ok,11673,187,6,2.071703
7,AMGN,ok,10850,187,6,1.905961
8,AMZN,ok,5882,187,6,1.083765
9,ANET,ok,3041,187,6,0.617896


Selected families (fundamentals + technicals):


,source,family,feature_count,strategy_source,is_technical
0,financetoolkit,ft_growth_balance,56,financetoolkit.ft_growth_balance,False
1,financetoolkit,ft_growth_cash,50,financetoolkit.ft_growth_cash,False
2,financetoolkit,ft_growth_income,38,financetoolkit.ft_growth_income,False
3,financetoolkit,ft_ratios_efficiency,5,financetoolkit.ft_ratios_efficiency,False
4,financetoolkit,ft_ratios_liquidity,7,financetoolkit.ft_ratios_liquidity,False
5,financetoolkit,ft_ratios_profitability,14,financetoolkit.ft_ratios_profitability,False
6,financetoolkit,ft_ratios_solvency,15,financetoolkit.ft_ratios_solvency,False
7,financetoolkit,ft_ratios_valuation,24,financetoolkit.ft_ratios_valuation,False
8,fmp,economic_indicators,36,fmp.economic_indicators,False
9,fmp,fmp_balance_mcap,55,fmp.fmp_balance_mcap,False


## Option Coverage By Symbol (Earliest Available Date)

Uses the **deduped** oracle windows (`oracle_trade_windows_unique`: one row per symbol/side/entry/exit).

Before labeling, read warehouse ThetaData cache coverage for each screened symbol:
- `min_snapshot_date`: earliest cached option chain date
- `max_snapshot_date`: latest cached option chain date

Windows with `entry_date` before that symbol's earliest option date (or with no option coverage) are filtered out up front. Windows with `exit_date` after the latest coverage date are also filtered. Remaining endpoint gaps are still skipped later during labeling.


In [7]:
from quant_warehouse.platforms.data_providers.thetadata.options import option_chain_coverage

if "oracle_trade_windows_unique" not in globals() or oracle_trade_windows_unique.empty:
    if "oracle_trade_windows" in globals() and not oracle_trade_windows.empty:
        raise RuntimeError("Oracle trades exist but unique windows were not built. Re-run the Oracle Trade Labels cell.")
    raise RuntimeError("Run the Oracle Trade Labels cell first.")
if "screened_equity_symbols" not in globals() or not screened_equity_symbols:
    raise RuntimeError("Run the Universe Screen cell first.")

if RUN_OPTION_COVERAGE:
    option_coverage = option_chain_coverage(tuple(screened_equity_symbols))
    if option_coverage.empty:
        option_coverage = pd.DataFrame(
            columns=["symbol", "row_count", "snapshot_day_count", "min_snapshot_date", "max_snapshot_date"]
        )
    option_coverage = option_coverage.copy()
    option_coverage["symbol"] = option_coverage["symbol"].astype(str).str.upper()
    option_coverage["min_snapshot_date"] = pd.to_datetime(option_coverage.get("min_snapshot_date"), errors="coerce").dt.normalize()
    option_coverage["max_snapshot_date"] = pd.to_datetime(option_coverage.get("max_snapshot_date"), errors="coerce").dt.normalize()
    option_coverage["has_option_data"] = (
        option_coverage["snapshot_day_count"].fillna(0).astype(int).gt(0)
        & option_coverage["min_snapshot_date"].notna()
    )

    OPTION_COVERAGE_PATH.parent.mkdir(parents=True, exist_ok=True)
    option_coverage.to_csv(OPTION_COVERAGE_PATH, index=False)

    coverage_lookup = option_coverage.set_index("symbol")[
        ["min_snapshot_date", "max_snapshot_date", "has_option_data", "snapshot_day_count"]
    ]

    trades = oracle_trade_windows_unique.copy()
    trades["symbol"] = trades["symbol"].astype(str).str.upper()
    trades["entry_date"] = pd.to_datetime(trades["entry_date"], errors="coerce").dt.normalize()
    trades["exit_date"] = pd.to_datetime(trades["exit_date"], errors="coerce").dt.normalize()
    trades = trades.merge(
        coverage_lookup.reset_index(),
        on="symbol",
        how="left",
        suffixes=("", "_coverage"),
    )
    trades["has_option_data"] = trades["has_option_data"].fillna(False).astype(bool)

    before_coverage_mask = (
        ~trades["has_option_data"]
        | trades["min_snapshot_date"].isna()
        | trades["entry_date"].lt(trades["min_snapshot_date"])
    )
    after_coverage_mask = trades["max_snapshot_date"].notna() & trades["exit_date"].gt(trades["max_snapshot_date"])
    # Keep trades fully inside [min_snapshot_date, max_snapshot_date] for the symbol.
    in_coverage_mask = (
        trades["has_option_data"]
        & trades["min_snapshot_date"].notna()
        & trades["max_snapshot_date"].notna()
        & trades["entry_date"].ge(trades["min_snapshot_date"])
        & trades["exit_date"].le(trades["max_snapshot_date"])
        & trades["exit_date"].gt(trades["entry_date"])
    )

    oracle_trades_before_option_coverage = trades.loc[before_coverage_mask | after_coverage_mask].copy()
    oracle_trades_before_option_coverage["skip_reason"] = np.select(
        [
            ~oracle_trades_before_option_coverage["has_option_data"].fillna(False),
            oracle_trades_before_option_coverage["entry_date"].lt(
                oracle_trades_before_option_coverage["min_snapshot_date"]
            ),
            oracle_trades_before_option_coverage["exit_date"].gt(
                oracle_trades_before_option_coverage["max_snapshot_date"]
            ),
        ],
        [
            "no_option_coverage_for_symbol",
            "entry_before_earliest_option_date",
            "exit_after_latest_option_date",
        ],
        default="outside_option_coverage",
    )
    oracle_trade_windows_in_option_coverage = trades.loc[in_coverage_mask].copy()

    oracle_trades_before_option_coverage.to_parquet(ORACLE_TRADES_BEFORE_COVERAGE_PATH, index=False)
    oracle_trade_windows_in_option_coverage.to_parquet(ORACLE_TRADES_IN_COVERAGE_PATH, index=False)

    print(
        {
            "symbols_screened": len(screened_equity_symbols),
            "symbols_with_option_data": int(option_coverage["has_option_data"].sum()),
            "oracle_trades_multi_k": int(len(oracle_trade_windows)) if "oracle_trade_windows" in globals() else None,
            "oracle_trades_unique": int(len(oracle_trade_windows_unique)),
            "oracle_trades_in_option_coverage": int(len(oracle_trade_windows_in_option_coverage)),
            "oracle_trades_outside_option_coverage": int(len(oracle_trades_before_option_coverage)),
            "coverage_path": str(OPTION_COVERAGE_PATH),
        }
    )
    display(
        option_coverage[
            [c for c in ["symbol", "has_option_data", "snapshot_day_count", "min_snapshot_date", "max_snapshot_date", "row_count"] if c in option_coverage.columns]
        ].sort_values(["has_option_data", "symbol"], ascending=[False, True])
    )
    if not oracle_trades_before_option_coverage.empty:
        display(
            oracle_trades_before_option_coverage.groupby(["symbol", "skip_reason"], dropna=False)
            .size()
            .rename("trades")
            .reset_index()
            .sort_values(["trades", "symbol"], ascending=[False, True])
            .head(40)
        )
    display(oracle_trade_windows_in_option_coverage.head(20))
else:
    if not ORACLE_TRADES_IN_COVERAGE_PATH.exists():
        raise FileNotFoundError(
            f"Missing in-coverage trades at {ORACLE_TRADES_IN_COVERAGE_PATH}. Set RUN_OPTION_COVERAGE=True."
        )
    option_coverage = pd.read_csv(OPTION_COVERAGE_PATH) if OPTION_COVERAGE_PATH.exists() else pd.DataFrame()
    oracle_trade_windows_in_option_coverage = pd.read_parquet(ORACLE_TRADES_IN_COVERAGE_PATH)
    print(f"Loaded in-coverage oracle trades: {len(oracle_trade_windows_in_option_coverage):,}")


{'symbols_screened': 118, 'symbols_with_option_data': 113, 'oracle_trades_multi_k': 57697, 'oracle_trades_unique': 36106, 'oracle_trades_in_option_coverage': 8850, 'oracle_trades_outside_option_coverage': 27256, 'coverage_path': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2/option_family_ranker/option_chain_coverage_by_symbol.csv'}


,symbol,has_option_data,snapshot_day_count,min_snapshot_date,max_snapshot_date,row_count
5,AAPL,True,2126,2012-06-19,2026-07-10,7006297
33,ABBV,True,255,2017-01-03,2026-07-10,454072
50,ABT,True,255,2017-01-03,2026-07-07,379390
34,ADI,True,375,2012-06-20,2026-07-10,450308
41,ADP,True,359,2012-06-19,2026-07-09,413912
...,...,...,...,...,...,...
113,BRK-A,False,0,NaT,NaT,0
114,BRK-B,False,0,NaT,NaT,0
115,SOJE,False,0,NaT,NaT,0
116,SOMN,False,0,NaT,NaT,0


,symbol,skip_reason,trades
19,BNY,entry_before_earliest_option_date,437
86,RTX,entry_before_earliest_option_date,437
47,HWM,entry_before_earliest_option_date,429
36,DIS,entry_before_earliest_option_date,420
65,MRK,entry_before_earliest_option_date,418
14,BA,entry_before_earliest_option_date,417
49,IBM,entry_before_earliest_option_date,416
78,PG,entry_before_earliest_option_date,416
55,KO,entry_before_earliest_option_date,415
40,GE,entry_before_earliest_option_date,414


,symbol,side,entry_date,exit_date,k_min,k_max,k_count,multi_k_rows,freq,entry_px,exit_px,ret_dec,ret_pct,hold_days,k_values,k,trade_id,min_snapshot_date,max_snapshot_date,has_option_data,snapshot_day_count
0,MRVL,long,2026-07-08,2026-07-09,3,3,1,1,YE,236.7300,241.0400,0.018206,1.82%,1,3,3,MRVL|long|20260708|20260709,2012-06-15,2026-07-09,True,377
1,AVGO,long,2026-07-07,2026-07-10,3,3,1,1,YE,372.9600,395.7000,0.060972,6.10%,3,3,3,AVGO|long|20260707|20260710,2012-07-03,2026-07-10,True,2079
2,BA,short,2026-07-07,2026-07-10,3,3,1,1,YE,230.4600,223.8400,0.028725,2.87%,3,3,3,BA|short|20260707|20260710,2017-01-03,2026-07-10,True,254
3,MCD,short,2026-07-07,2026-07-09,3,3,1,1,YE,281.3800,277.5800,0.013505,1.35%,2,3,3,MCD|short|20260707|20260709,2020-03-18,2026-07-09,True,169
4,PEP,short,2026-07-07,2026-07-10,2,3,2,2,YE,144.5400,137.9600,0.045524,4.55%,3,"2,3",2,PEP|short|20260707|20260710,2017-01-04,2026-07-10,True,260
5,SNDK,long,2026-07-07,2026-07-09,2,3,2,2,YE,"1,638.8800","1,801.0000",0.098921,9.89%,2,"2,3",2,SNDK|long|20260707|20260709,2025-03-19,2026-07-09,True,46
6,CVX,long,2026-07-01,2026-07-08,3,3,1,1,YE,167.4700,174.9500,0.044665,4.47%,7,3,3,CVX|long|20260701|20260708,2017-01-04,2026-07-08,True,257
7,DE,short,2026-07-01,2026-07-10,3,3,1,1,YE,619.3000,599.4400,0.032068,3.21%,9,3,3,DE|short|20260701|20260710,2017-01-03,2026-07-10,True,261
8,HD,short,2026-07-01,2026-07-08,3,3,1,1,YE,350.8400,337.9900,0.036626,3.66%,7,3,3,HD|short|20260701|20260708,2017-01-06,2026-07-08,True,257
9,AMAT,short,2026-06-30,2026-07-07,1,3,3,3,YE,696.0000,558.2800,0.197874,19.79%,7,"1,2,3",1,AMAT|short|20260630|20260707,2012-06-20,2026-07-09,True,1244


## Option Labels: Entry Filters + Exit Existence

Uses oracle trades already filtered to each symbol's option coverage window (`oracle_trade_windows_in_option_coverage`).

**Entry-date filters only** (candidate selection):
1. Load cached ThetaData EOD chain for exact `entry_date` (`download_missing=False`).
2. Keep **calls** when `side == long`, **puts** when `side == short`.
3. Require usable **bid** and **ask** on entry (`ask > 0`, `ask >= bid`).
4. Require **max DTE <= 90** measured from the oracle trade **entry date** (`expiration - entry_date`).

**Exit date** (pricing only):
5. Load cached chain for exact `exit_date`.
6. Do **not** apply type / DTE / bid-ask candidate filters on the exit chain.
7. Keep entry candidates whose contract **exists** on the exit snapshot (inner join by contract id).
8. Price return as **buy entry ask / sell exit bid**:
   - `option_return = (exit_bid - entry_ask) / entry_ask`
   - No mid quotes, no intermediate-day marks.
9. Skip the trade if entry or exit historical data is missing, or no entry candidates survive filters, or none exist at exit.

This cell never downloads option data or stops a ThetaData job.


In [ ]:
from quant_warehouse.platforms.data_providers.thetadata.target_engineering import (
    OracleOptionLabelPanelSpec,
    build_oracle_option_label_panel,
)

if "oracle_trade_windows_in_option_coverage" not in globals() or oracle_trade_windows_in_option_coverage.empty:
    if "oracle_trade_windows" in globals() and not oracle_trade_windows.empty:
        raise RuntimeError(
            "Run the Option Coverage By Symbol cell first so trades are pre-filtered by earliest/latest option dates."
        )
    raise RuntimeError("No in-coverage oracle trades available. Run Oracle Trade Labels + Option Coverage cells first.")

if RUN_OPTION_LABEL_PANEL:
    option_label_result = build_oracle_option_label_panel(
        oracle_trade_windows_in_option_coverage,
        spec=OracleOptionLabelPanelSpec(
            max_trades=MAX_ORACLE_TRADES_FOR_OPTIONS,
            max_dte=OPTION_MAX_DTE,
            target_dte=OPTION_TARGET_DTE,
            progress_every=OPTION_PANEL_PROGRESS_EVERY,
        ),
        progress_logger=print,
    )
    option_candidate_panel = option_label_result.panel
    option_label_summary = option_label_result.summary
    OPTION_PANEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not option_candidate_panel.empty:
        option_candidate_panel.to_parquet(OPTION_PANEL_PATH, index=False)

    skipped_missing_df = pd.DataFrame(option_label_summary.get("skipped_missing_options") or [])
    labeled_ids = {str(x) for x in (option_label_summary.get("labeled_trade_ids") or []) if x is not None}
    parent_trades = oracle_trade_windows_in_option_coverage
    if "trade_id" in parent_trades.columns:
        trade_ids = parent_trades["trade_id"].astype(str)
        oracle_trades_labeled = parent_trades.loc[trade_ids.isin(labeled_ids)].copy()
        if not skipped_missing_df.empty and "trade_id" in skipped_missing_df.columns:
            skipped_ids = set(skipped_missing_df["trade_id"].dropna().astype(str))
            oracle_trades_skipped_missing = parent_trades.loc[trade_ids.isin(skipped_ids)].copy()
        else:
            oracle_trades_skipped_missing = parent_trades.iloc[0:0].copy()
    else:
        oracle_trades_labeled = parent_trades.iloc[0:0].copy()
        oracle_trades_skipped_missing = parent_trades.iloc[0:0].copy()

    oracle_trades_labeled.to_parquet(ORACLE_TRADES_LABELED_PATH, index=False)
    if not skipped_missing_df.empty:
        skipped_missing_df.to_parquet(ORACLE_TRADES_SKIPPED_PATH, index=False)
    else:
        pd.DataFrame(columns=["trade_id", "symbol", "side", "entry_date", "exit_date", "skip_reason"]).to_parquet(
            ORACLE_TRADES_SKIPPED_PATH, index=False
        )

    # Keep full skip lists on disk; trim from printed summary for readability.
    summary_for_json = {
        k: v
        for k, v in option_label_summary.items()
        if k not in {"skipped_missing_options", "labeled_trade_ids"}
    }
    summary_for_json["skipped_missing_options_path"] = str(ORACLE_TRADES_SKIPPED_PATH)
    summary_for_json["labeled_trades_path"] = str(ORACLE_TRADES_LABELED_PATH)
    summary_for_json["labeled_trade_count"] = int(len(oracle_trades_labeled))
    summary_for_json["skipped_missing_options_count"] = int(len(skipped_missing_df))
    OPTION_LABEL_SUMMARY_PATH.write_text(json.dumps(summary_for_json, indent=2, default=str), encoding="utf-8")
    print(json.dumps(summary_for_json, indent=2, default=str))
    print(
        f"Skipped {summary_for_json['skipped_missing_options_count']:,} oracle trades with missing historical option data "
        f"(entry and/or exit chain absent in warehouse cache)."
    )
    if not skipped_missing_df.empty:
        display(skipped_missing_df.head(20))
    if option_candidate_panel.empty:
        print("No option label rows were produced from cached entry/exit chains.")
    else:
        print(f"Wrote option panel: {OPTION_PANEL_PATH} rows={len(option_candidate_panel):,}")
        print(f"Wrote labeled oracle trades: {ORACLE_TRADES_LABELED_PATH} rows={len(oracle_trades_labeled):,}")
        display(
            option_candidate_panel.groupby(["side", "option_type"], dropna=False)
            .size()
            .rename("rows")
            .reset_index()
        )
        display(
            option_candidate_panel[
                [
                    c
                    for c in [
                        "trade_id",
                        "symbol",
                        "side",
                        "option_type",
                        "option_action",
                        "entry_date",
                        "equity_exit_date",
                        "contract_symbol",
                        "strike",
                        "option_return",
                        "rank_y",
                        "dte",
                        "abs_moneyness",
                    ]
                    if c in option_candidate_panel.columns
                ]
            ].head(20)
        )
else:
    if not OPTION_PANEL_PATH.exists():
        raise FileNotFoundError(f"Missing option panel at {OPTION_PANEL_PATH}")
    option_candidate_panel = pd.read_parquet(OPTION_PANEL_PATH)
    print(f"Loaded option panel from {OPTION_PANEL_PATH}: {len(option_candidate_panel):,} rows")


[option-labels] done=50/8850 labeled=50 rows=6400 skipped_missing_options=0 empty_intersection=0 symbol=MRVL
[option-labels] done=100/8850 labeled=100 rows=12800 skipped_missing_options=0 empty_intersection=0 symbol=MRVL
[option-labels] done=150/8850 labeled=150 rows=19108 skipped_missing_options=0 empty_intersection=0 symbol=AVGO
[option-labels] done=200/8850 labeled=200 rows=25508 skipped_missing_options=0 empty_intersection=0 symbol=AVGO
[option-labels] done=250/8850 labeled=250 rows=31076 skipped_missing_options=0 empty_intersection=0 symbol=BA


## Equity Family Models + Option Meta-Stack

Reuses **trading_app_v2** pieces:

1. **Feature engineering** (previous cell): `feature_panel` + `selected_feature_metadata`
2. **Stage A:** `_train_family_models` on **equity** `oracle_label_rows` for **all selected families** (fundamentals + technicals when enabled)
3. Score families on option **entry dates** → pivot to wide family score columns
4. **Stage B:** train option ranker on **family scores + option features** (`rank_y`, entry ask → exit bid)
5. Forward eval vs `fixed_near_atm`

Temporal split by option/oracle **entry_date**: train ≤ `OPTION_TRAIN_END`, forward ≥ `OPTION_EVAL_START`.

### Comparison output (`fixed_near_atm` vs `ml_options`)

After Stage B, the cell shows a **forward-only** (entry ≥ eval start) comparison:

| Block | What you get |
|-------|----------------|
| **Headline** | One verdict card: mean/median return, win rate, rank_y + absolute & % lift |
| **Aggregate metrics** | Side-by-side table + Δ and % lift rows |
| **Head-to-head** | Per-trade win counts, same-contract rate, lift when picks disagree |
| **Contract choice** | Mean DTE / moneyness / delta / IV of ML vs fixed *when different* |
| **By side / month** | Long vs short and monthly mean returns + lift |
| **Best / worst** | Top & bottom 15 trades by return lift |
| **Charts** | Metric bars, return densities, lift hist, scatter, cumulative return, monthly lift |
| **CSVs** | Written under the meta-stack artifact dir (`fixed_vs_ml_options_*.csv` + `.png`) |

- `fixed_near_atm` — soft score nearest ATM / ~90 DTE
- `ml_options` — meta-stack ranker (equity family scores + option features)


In [ ]:
from __future__ import annotations

import gc
import pickle
from time import perf_counter

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from quant_orchestrator.research_tools.ml_trading_experiment import (
    MLTradingExperimentConfig,
    _score_family_models,
    _train_family_models,
)
from quant_orchestrator.research_tools.option_family_ranker import (
    OPTION_FEATURES,
    _evaluate_selector,
    _filter_oracle_entry_options,
    _select_diverse_option_candidates,
    _training_and_scoring_frames,
)
from quant_orchestrator.research_tools.option_meta_ranker import (
    OPTION_TARGET_CONTRACT,
    OptionMetaRankerConfig,
    _fit_option_ranker,
    _predict_option_ranker,
)

option_train_end_value = optional_date(OPTION_TRAIN_END) or "2024-12-31"
option_eval_start_value = optional_date(OPTION_EVAL_START) or "2025-01-01"
train_end_ts = pd.Timestamp(option_train_end_value).normalize()
eval_start_ts = pd.Timestamp(option_eval_start_value).normalize()
if eval_start_ts <= train_end_ts:
    raise ValueError("eval_start must be after train_end")

META_DIR = paths.option_artifact_dir / "option_meta_stack"
META_DIR.mkdir(parents=True, exist_ok=True)

def _select_top1_per_trade(frame: pd.DataFrame, *, score_col: str, model_name: str) -> pd.DataFrame:
    """One selected option contract per trade_id by highest score."""
    work = frame.copy()
    work[score_col] = pd.to_numeric(work[score_col], errors="coerce")
    work["option_return"] = pd.to_numeric(work["option_return"], errors="coerce")
    if "rank_y" in work.columns:
        work["rank_y"] = pd.to_numeric(work["rank_y"], errors="coerce")
    keep = work.dropna(subset=[score_col, "trade_id"]).sort_values(
        ["trade_id", score_col], ascending=[True, False], kind="stable"
    )
    picks = keep.groupby("trade_id", as_index=False, sort=False).head(1).copy()
    picks["model"] = model_name
    picks["score"] = picks[score_col]
    return picks.reset_index(drop=True)


def _model_trade_metrics(picks: pd.DataFrame, *, model_name: str) -> dict:
    if picks is None or picks.empty:
        return {"model": model_name, "trades": 0}
    rets = pd.to_numeric(picks["option_return"], errors="coerce")
    out = {
        "model": model_name,
        "trades": int(len(picks)),
        "mean_option_return": float(rets.mean()),
        "median_option_return": float(rets.median()),
        "win_rate": float(rets.gt(0).mean()),
        "p25_option_return": float(rets.quantile(0.25)),
        "p75_option_return": float(rets.quantile(0.75)),
        "std_option_return": float(rets.std(ddof=0)) if len(rets) else float("nan"),
    }
    if "rank_y" in picks.columns:
        ranks = pd.to_numeric(picks["rank_y"], errors="coerce")
        out["mean_rank_y"] = float(ranks.mean())
        out["median_rank_y"] = float(ranks.median())
    return out


def display_fixed_vs_ml_options_comparison(
    forward_frame: pd.DataFrame,
    *,
    fixed_score_col: str = "fixed_near_atm_score",
    ml_score_col: str = "pred_meta_stack_rank",
    fixed_name: str = "fixed_near_atm",
    ml_name: str = "ml_options",
    output_dir: Path | None = None,
) -> dict:
    """Rich side-by-side forward comparison of baseline vs ML option picker.

    Shows: headline deltas, aggregate metrics with % lift, head-to-head,
    contract-choice diffs (when models disagree), monthly & side breakdowns,
    best/worst trades, equity curves + return scatter.
    """
    from IPython.display import HTML, display as ipy_display

    def _disp(obj):
        try:
            ipy_display(obj)
        except Exception:
            display(obj)

    print("=" * 78)
    print("FORWARD COMPARISON:  fixed_near_atm  vs  ml_options (meta-stack)")
    print("=" * 78)

    fixed_picks = _select_top1_per_trade(forward_frame, score_col=fixed_score_col, model_name=fixed_name)
    ml_picks = _select_top1_per_trade(forward_frame, score_col=ml_score_col, model_name=ml_name)

    metrics = pd.DataFrame(
        [
            _model_trade_metrics(fixed_picks, model_name=fixed_name),
            _model_trade_metrics(ml_picks, model_name=ml_name),
        ]
    ).set_index("model")

    # Absolute deltas + relative % lift on key return/rank metrics
    abs_delta = {}
    pct_lift = {}
    for col in metrics.columns:
        if col == "trades":
            continue
        try:
            f_v = float(metrics.loc[fixed_name, col])
            m_v = float(metrics.loc[ml_name, col])
            abs_delta[col] = m_v - f_v
            if abs(f_v) > 1e-12:
                pct_lift[col] = (m_v - f_v) / abs(f_v)
            else:
                pct_lift[col] = float("nan")
        except Exception:
            abs_delta[col] = float("nan")
            pct_lift[col] = float("nan")

    # ----- 0) Headline -----
    mean_f = float(metrics.loc[fixed_name, "mean_option_return"])
    mean_m = float(metrics.loc[ml_name, "mean_option_return"])
    med_f = float(metrics.loc[fixed_name, "median_option_return"])
    med_m = float(metrics.loc[ml_name, "median_option_return"])
    wr_f = float(metrics.loc[fixed_name, "win_rate"])
    wr_m = float(metrics.loc[ml_name, "win_rate"])
    rank_f = float(metrics.loc[fixed_name, "mean_rank_y"]) if "mean_rank_y" in metrics.columns else float("nan")
    rank_m = float(metrics.loc[ml_name, "mean_rank_y"]) if "mean_rank_y" in metrics.columns else float("nan")
    n_trades = int(metrics.loc[fixed_name, "trades"])
    mean_lift = mean_m - mean_f
    mean_lift_pct = (mean_m - mean_f) / abs(mean_f) if abs(mean_f) > 1e-12 else float("nan")
    verdict = "BEATS" if mean_lift > 0 else ("TRAILS" if mean_lift < 0 else "TIES")
    color = "#198754" if mean_lift > 0 else ("#dc3545" if mean_lift < 0 else "#6c757d")

    headline_html = f"""
    <div style="border:2px solid {color}; border-radius:10px; padding:14px 18px; margin:8px 0 14px 0;
                background:linear-gradient(90deg, #f8f9fa 0%, #ffffff 100%); font-family:system-ui,sans-serif;">
      <div style="font-size:13px; color:#6c757d; letter-spacing:0.04em; text-transform:uppercase;">
        Forward only · one contract per trade · n={n_trades}
      </div>
      <div style="font-size:22px; font-weight:700; margin-top:4px; color:{color};">
        ml_options {verdict} fixed_near_atm
      </div>
      <div style="display:flex; flex-wrap:wrap; gap:18px; margin-top:10px; font-size:15px;">
        <div><b>Mean return</b><br>
          fixed <code>{mean_f:.3f}</code> → ml <code>{mean_m:.3f}</code>
          <span style="color:{color}; font-weight:600;">({mean_lift:+.3f} / {mean_lift_pct:+.0%})</span>
        </div>
        <div><b>Median return</b><br>
          fixed <code>{med_f:.3f}</code> → ml <code>{med_m:.3f}</code>
          <span style="color:{color}; font-weight:600;">({med_m - med_f:+.3f})</span>
        </div>
        <div><b>Win rate</b><br>
          fixed <code>{wr_f:.1%}</code> → ml <code>{wr_m:.1%}</code>
          <span style="font-weight:600;">({wr_m - wr_f:+.1%})</span>
        </div>
        <div><b>Mean rank_y</b><br>
          fixed <code>{rank_f:.3f}</code> → ml <code>{rank_m:.3f}</code>
          <span style="font-weight:600;">({rank_m - rank_f:+.3f})</span>
        </div>
      </div>
      <div style="margin-top:8px; font-size:12px; color:#6c757d;">
        Positive lift = ML picked a higher option_return contract on the same trade.
        rank_y ∈ [0,1] = within-trade percentile of option_return (higher is better).
      </div>
    </div>
    """
    _disp(HTML(headline_html))

    # ----- 1) Aggregate metrics with deltas -----
    preferred = [
        "trades",
        "mean_option_return",
        "median_option_return",
        "win_rate",
        "mean_rank_y",
        "median_rank_y",
        "p25_option_return",
        "p75_option_return",
        "std_option_return",
    ]
    comparison = metrics[[c for c in preferred if c in metrics.columns]].copy()
    delta_row = pd.DataFrame([{"model": "Δ ml − fixed", **abs_delta}]).set_index("model")
    lift_row = pd.DataFrame([{"model": "% lift (ml vs |fixed|)", **pct_lift}]).set_index("model")
    comparison = pd.concat([comparison, delta_row, lift_row], axis=0)
    comparison = comparison[[c for c in preferred if c in comparison.columns]]

    print("\n1) Aggregate forward metrics (one pick per trade)")
    fmt = {c: "{:.4f}" for c in comparison.columns if c != "trades"}
    fmt["trades"] = "{:.0f}"
    try:
        styled = comparison.style.format(fmt, na_rep="—")
        # highlight delta row
        def _hl_delta(row):
            if str(row.name).startswith("Δ"):
                return [
                    f"background-color: {'#d1e7dd' if (isinstance(v, (int, float)) and v > 0) else ('#f8d7da' if (isinstance(v, (int, float)) and v < 0) else '#fff3cd')}"
                    for v in row
                ]
            if str(row.name).startswith("%"):
                return ["background-color: #e7f1ff"] * len(row)
            return [""] * len(row)

        _disp(styled.apply(_hl_delta, axis=1))
    except Exception:
        _disp(comparison)

    # Side-by-side wide table for slides / quick scan
    side_by_side = pd.DataFrame(
        {
            "metric": [c for c in preferred if c in metrics.columns and c != "trades"],
            fixed_name: [metrics.loc[fixed_name, c] for c in preferred if c in metrics.columns and c != "trades"],
            ml_name: [metrics.loc[ml_name, c] for c in preferred if c in metrics.columns and c != "trades"],
        }
    )
    side_by_side["Δ (ml − fixed)"] = side_by_side[ml_name] - side_by_side[fixed_name]
    side_by_side["% lift"] = side_by_side.apply(
        lambda r: (r[ml_name] - r[fixed_name]) / abs(r[fixed_name]) if abs(r[fixed_name]) > 1e-12 else float("nan"),
        axis=1,
    )
    print("\n   Side-by-side (easier scan):")
    _disp(
        side_by_side.set_index("metric").style.format(
            {
                fixed_name: "{:.4f}",
                ml_name: "{:.4f}",
                "Δ (ml − fixed)": "{:+.4f}",
                "% lift": "{:+.1%}",
            }
        )
    )

    # ----- 2) Paired trade-level -----
    f = fixed_picks.rename(
        columns={
            "option_return": "fixed_option_return",
            "rank_y": "fixed_rank_y",
            "score": "fixed_score",
            "contract_symbol": "fixed_contract",
            "dte": "fixed_dte",
            "moneyness": "fixed_moneyness",
            "abs_moneyness": "fixed_abs_moneyness",
            "delta": "fixed_delta",
            "abs_delta": "fixed_abs_delta",
            "iv": "fixed_iv",
            "strike": "fixed_strike",
            "expiration": "fixed_expiration",
            "option_type": "fixed_option_type",
        }
    )
    m = ml_picks.rename(
        columns={
            "option_return": "ml_option_return",
            "rank_y": "ml_rank_y",
            "score": "ml_score",
            "contract_symbol": "ml_contract",
            "dte": "ml_dte",
            "moneyness": "ml_moneyness",
            "abs_moneyness": "ml_abs_moneyness",
            "delta": "ml_delta",
            "abs_delta": "ml_abs_delta",
            "iv": "ml_iv",
            "strike": "ml_strike",
            "expiration": "ml_expiration",
            "option_type": "ml_option_type",
        }
    )
    keep_f = [
        c
        for c in [
            "trade_id",
            "symbol",
            "entry_date",
            "side",
            "equity_signal_side",
            "fixed_option_return",
            "fixed_rank_y",
            "fixed_score",
            "fixed_contract",
            "fixed_dte",
            "fixed_moneyness",
            "fixed_abs_moneyness",
            "fixed_delta",
            "fixed_abs_delta",
            "fixed_iv",
            "fixed_strike",
            "fixed_expiration",
            "fixed_option_type",
        ]
        if c in f.columns
    ]
    keep_m = [
        c
        for c in [
            "trade_id",
            "ml_option_return",
            "ml_rank_y",
            "ml_score",
            "ml_contract",
            "ml_dte",
            "ml_moneyness",
            "ml_abs_moneyness",
            "ml_delta",
            "ml_abs_delta",
            "ml_iv",
            "ml_strike",
            "ml_expiration",
            "ml_option_type",
        ]
        if c in m.columns
    ]
    paired = f[keep_f].merge(m[keep_m], on="trade_id", how="inner")
    paired["return_lift_ml_minus_fixed"] = (
        pd.to_numeric(paired["ml_option_return"], errors="coerce")
        - pd.to_numeric(paired["fixed_option_return"], errors="coerce")
    )
    if "fixed_rank_y" in paired.columns and "ml_rank_y" in paired.columns:
        paired["rank_lift_ml_minus_fixed"] = (
            pd.to_numeric(paired["ml_rank_y"], errors="coerce")
            - pd.to_numeric(paired["fixed_rank_y"], errors="coerce")
        )
    same_contract = False
    if "fixed_contract" in paired.columns and "ml_contract" in paired.columns:
        paired["same_contract"] = paired["fixed_contract"].astype(str).eq(paired["ml_contract"].astype(str))
        same_contract = True
    for a, b, out_c in [
        ("fixed_dte", "ml_dte", "dte_diff_ml_minus_fixed"),
        ("fixed_abs_moneyness", "ml_abs_moneyness", "abs_moneyness_diff_ml_minus_fixed"),
        ("fixed_abs_delta", "ml_abs_delta", "abs_delta_diff_ml_minus_fixed"),
        ("fixed_iv", "ml_iv", "iv_diff_ml_minus_fixed"),
    ]:
        if a in paired.columns and b in paired.columns:
            paired[out_c] = pd.to_numeric(paired[b], errors="coerce") - pd.to_numeric(paired[a], errors="coerce")

    lift = pd.to_numeric(paired["return_lift_ml_minus_fixed"], errors="coerce")
    disagree = paired.loc[~paired["same_contract"]] if same_contract else paired
    head_to_head = pd.DataFrame(
        [
            {
                "paired_trades": int(len(paired)),
                "ml_better_return_count": int(lift.gt(0).sum()),
                "fixed_better_return_count": int(lift.lt(0).sum()),
                "tie_return_count": int(lift.eq(0).sum()),
                "ml_win_rate_vs_fixed": float(lift.gt(0).mean()) if len(lift) else float("nan"),
                "mean_return_lift": float(lift.mean()) if len(lift) else float("nan"),
                "median_return_lift": float(lift.median()) if len(lift) else float("nan"),
                "pct_same_contract": float(paired["same_contract"].mean()) if same_contract else float("nan"),
                "pct_different_contract": float((~paired["same_contract"]).mean()) if same_contract else float("nan"),
                "mean_return_lift_when_disagree": float(
                    pd.to_numeric(disagree["return_lift_ml_minus_fixed"], errors="coerce").mean()
                )
                if len(disagree)
                else float("nan"),
            }
        ]
    )
    print("\n2) Head-to-head on the same forward trades")
    h2h_view = head_to_head.T.rename(columns={0: "value"})
    _disp(h2h_view.style.format({"value": lambda v: f"{v:.4f}" if isinstance(v, float) else str(v)}))

    # ----- 3) Lift distribution -----
    print("\n3) Return lift distribution (ml_options − fixed_near_atm)")
    lift_summary = lift.describe(percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]).to_frame("return_lift")
    _disp(lift_summary.style.format("{:.4f}"))

    # ----- 4) Contract-choice differences when models disagree -----
    print("\n4) How ML contracts differ when pick ≠ fixed_near_atm")
    if same_contract and len(disagree):
        choice_rows = []
        for label, col_f, col_m in [
            ("dte", "fixed_dte", "ml_dte"),
            ("abs_moneyness", "fixed_abs_moneyness", "ml_abs_moneyness"),
            ("abs_delta", "fixed_abs_delta", "ml_abs_delta"),
            ("iv", "fixed_iv", "ml_iv"),
            ("option_return", "fixed_option_return", "ml_option_return"),
            ("rank_y", "fixed_rank_y", "ml_rank_y"),
        ]:
            if col_f in disagree.columns and col_m in disagree.columns:
                fv = pd.to_numeric(disagree[col_f], errors="coerce")
                mv = pd.to_numeric(disagree[col_m], errors="coerce")
                choice_rows.append(
                    {
                        "feature": label,
                        "fixed_mean": float(fv.mean()),
                        "ml_mean": float(mv.mean()),
                        "Δ mean (ml − fixed)": float((mv - fv).mean()),
                        "n_disagree": int(len(disagree)),
                    }
                )
        choice_df = pd.DataFrame(choice_rows)
        _disp(
            choice_df.set_index("feature").style.format(
                {
                    "fixed_mean": "{:.4f}",
                    "ml_mean": "{:.4f}",
                    "Δ mean (ml − fixed)": "{:+.4f}",
                    "n_disagree": "{:.0f}",
                }
            )
        )
        print(
            f"   → Models pick the same contract on {paired['same_contract'].mean():.1%} of trades; "
            f"on the other {(~paired['same_contract']).mean():.1%}, mean return lift = "
            f"{pd.to_numeric(disagree['return_lift_ml_minus_fixed'], errors='coerce').mean():+.3f}"
        )
    else:
        choice_df = pd.DataFrame()
        print("   (no contract columns or all picks identical)")

    # ----- 5) Breakdowns: by side & by month -----
    print("\n5) Breakdown by equity side / option type")
    side_col = "side" if "side" in paired.columns else ("equity_signal_side" if "equity_signal_side" in paired.columns else None)
    if side_col is not None:
        by_side = (
            paired.groupby(side_col, dropna=False)
            .agg(
                trades=("trade_id", "count"),
                fixed_mean_ret=("fixed_option_return", "mean"),
                ml_mean_ret=("ml_option_return", "mean"),
                mean_lift=("return_lift_ml_minus_fixed", "mean"),
                median_lift=("return_lift_ml_minus_fixed", "median"),
                ml_win_rate=("return_lift_ml_minus_fixed", lambda s: float(pd.to_numeric(s, errors="coerce").gt(0).mean())),
            )
            .reset_index()
        )
        _disp(
            by_side.style.format(
                {
                    "fixed_mean_ret": "{:.4f}",
                    "ml_mean_ret": "{:.4f}",
                    "mean_lift": "{:+.4f}",
                    "median_lift": "{:+.4f}",
                    "ml_win_rate": "{:.1%}",
                }
            )
        )
    else:
        by_side = pd.DataFrame()
        print("   (no side column)")

    print("\n6) Monthly mean return (fixed vs ml) + lift")
    paired["_entry_month"] = pd.to_datetime(paired["entry_date"], errors="coerce").dt.to_period("M").astype(str)
    by_month = (
        paired.groupby("_entry_month", dropna=False)
        .agg(
            trades=("trade_id", "count"),
            fixed_mean_ret=("fixed_option_return", "mean"),
            ml_mean_ret=("ml_option_return", "mean"),
            mean_lift=("return_lift_ml_minus_fixed", "mean"),
            ml_win_rate=("return_lift_ml_minus_fixed", lambda s: float(pd.to_numeric(s, errors="coerce").gt(0).mean())),
        )
        .reset_index()
        .rename(columns={"_entry_month": "month"})
        .sort_values("month")
    )
    _disp(
        by_month.style.format(
            {
                "fixed_mean_ret": "{:.4f}",
                "ml_mean_ret": "{:.4f}",
                "mean_lift": "{:+.4f}",
                "ml_win_rate": "{:.1%}",
            }
        )
    )

    # ----- 7) Best / worst -----
    show_cols = [
        c
        for c in [
            "trade_id",
            "symbol",
            "entry_date",
            "side",
            "fixed_contract",
            "ml_contract",
            "fixed_dte",
            "ml_dte",
            "fixed_abs_moneyness",
            "ml_abs_moneyness",
            "fixed_option_return",
            "ml_option_return",
            "return_lift_ml_minus_fixed",
            "fixed_rank_y",
            "ml_rank_y",
            "same_contract",
        ]
        if c in paired.columns
    ]
    print("\n7) Largest ML improvements (top 15 by return lift)")
    _disp(paired.sort_values("return_lift_ml_minus_fixed", ascending=False)[show_cols].head(15))
    print("\n8) Largest ML underperformance (bottom 15 by return lift)")
    _disp(paired.sort_values("return_lift_ml_minus_fixed", ascending=True)[show_cols].head(15))

    # ----- 9) Charts -----
    try:
        import matplotlib
        matplotlib.use("Agg")  # headless-safe; avoid hang on plt.show() in scripts
        import matplotlib.pyplot as plt

        fig = plt.figure(figsize=(16, 10))
        gs = fig.add_gridspec(2, 3, hspace=0.32, wspace=0.28)

        # mean / median / win-rate bars
        ax0 = fig.add_subplot(gs[0, 0])
        x = np.arange(3)
        w = 0.35
        fixed_vals = [mean_f, med_f, wr_f]
        ml_vals = [mean_m, med_m, wr_m]
        # scale win_rate for display note — keep raw, different units ok with labels
        ax0.bar(x - w / 2, fixed_vals, w, label=fixed_name, color="#6c757d")
        ax0.bar(x + w / 2, ml_vals, w, label=ml_name, color="#0d6efd")
        ax0.set_xticks(x)
        ax0.set_xticklabels(["mean ret", "median ret", "win rate"])
        ax0.axhline(0, color="black", lw=0.8)
        ax0.set_title("Key metrics side-by-side")
        ax0.legend(fontsize=8)

        # return distributions overlay
        ax1 = fig.add_subplot(gs[0, 1])
        fr = pd.to_numeric(paired["fixed_option_return"], errors="coerce").dropna()
        mr = pd.to_numeric(paired["ml_option_return"], errors="coerce").dropna()
        lo = float(min(fr.quantile(0.01), mr.quantile(0.01)))
        hi = float(max(fr.quantile(0.99), mr.quantile(0.99)))
        bins = np.linspace(lo, hi, 40)
        ax1.hist(fr, bins=bins, alpha=0.55, label=fixed_name, color="#6c757d", density=True)
        ax1.hist(mr, bins=bins, alpha=0.55, label=ml_name, color="#0d6efd", density=True)
        ax1.axvline(mean_f, color="#6c757d", ls="--", lw=1.2)
        ax1.axvline(mean_m, color="#0d6efd", ls="--", lw=1.2)
        ax1.set_title("Return distribution (density)")
        ax1.legend(fontsize=8)

        # lift hist
        ax2 = fig.add_subplot(gs[0, 2])
        ax2.hist(lift.dropna(), bins=40, color="#198754", alpha=0.85)
        ax2.axvline(0, color="black", lw=1)
        ax2.axvline(float(lift.mean()), color="#dc3545", ls="--", label=f"mean={lift.mean():.3f}")
        ax2.axvline(float(lift.median()), color="#fd7e14", ls=":", label=f"med={lift.median():.3f}")
        ax2.set_title("Return lift (ml − fixed)")
        ax2.legend(fontsize=8)

        # scatter fixed vs ml returns
        ax3 = fig.add_subplot(gs[1, 0])
        ax3.scatter(
            pd.to_numeric(paired["fixed_option_return"], errors="coerce"),
            pd.to_numeric(paired["ml_option_return"], errors="coerce"),
            s=12,
            alpha=0.35,
            c=np.where(lift.gt(0), "#198754", np.where(lift.lt(0), "#dc3545", "#6c757d")),
        )
        lims = [
            min(ax3.get_xlim()[0], ax3.get_ylim()[0]),
            max(ax3.get_xlim()[1], ax3.get_ylim()[1]),
        ]
        ax3.plot(lims, lims, "k--", lw=0.9, label="y = x (tie)")
        ax3.set_xlabel(f"{fixed_name} return")
        ax3.set_ylabel(f"{ml_name} return")
        ax3.set_title("Per-trade returns (green = ML better)")
        ax3.legend(fontsize=8)

        # cumulative sum of returns by entry_date order
        ax4 = fig.add_subplot(gs[1, 1])
        chron = paired.sort_values("entry_date").copy()
        chron["cum_fixed"] = pd.to_numeric(chron["fixed_option_return"], errors="coerce").cumsum()
        chron["cum_ml"] = pd.to_numeric(chron["ml_option_return"], errors="coerce").cumsum()
        ax4.plot(chron["entry_date"], chron["cum_fixed"], label=fixed_name, color="#6c757d", lw=1.5)
        ax4.plot(chron["entry_date"], chron["cum_ml"], label=ml_name, color="#0d6efd", lw=1.5)
        ax4.fill_between(
            chron["entry_date"],
            chron["cum_fixed"],
            chron["cum_ml"],
            where=chron["cum_ml"] >= chron["cum_fixed"],
            color="#198754",
            alpha=0.15,
            interpolate=True,
        )
        ax4.fill_between(
            chron["entry_date"],
            chron["cum_fixed"],
            chron["cum_ml"],
            where=chron["cum_ml"] < chron["cum_fixed"],
            color="#dc3545",
            alpha=0.15,
            interpolate=True,
        )
        ax4.set_title("Cumulative option return (sum, by entry_date)")
        ax4.legend(fontsize=8)
        ax4.tick_params(axis="x", rotation=30)

        # monthly mean lift bars
        ax5 = fig.add_subplot(gs[1, 2])
        if len(by_month):
            colors = ["#198754" if v >= 0 else "#dc3545" for v in by_month["mean_lift"]]
            ax5.bar(by_month["month"].astype(str), by_month["mean_lift"], color=colors)
            ax5.axhline(0, color="black", lw=0.8)
            ax5.set_title("Monthly mean return lift")
            ax5.tick_params(axis="x", rotation=45, labelsize=7)
        else:
            ax5.set_visible(False)

        fig.suptitle(
            f"Forward: fixed_near_atm vs ml_options  |  mean lift {mean_lift:+.3f} ({mean_lift_pct:+.0%})",
            fontsize=13,
            fontweight="bold",
            y=1.01,
        )
        if output_dir is not None:
            fig_path = Path(output_dir) / "fixed_vs_ml_options_comparison.png"
            fig.savefig(fig_path, dpi=130, bbox_inches="tight")
            print(f"\nSaved chart: {fig_path}")
        try:
            plt.show()
        except Exception:
            pass
        plt.close(fig)
    except Exception as exc:
        print(f"(plot skipped: {exc})")

    if output_dir is not None:
        out = Path(output_dir)
        comparison.reset_index().to_csv(out / "fixed_vs_ml_options_metrics.csv", index=False)
        side_by_side.to_csv(out / "fixed_vs_ml_options_side_by_side.csv", index=False)
        head_to_head.to_csv(out / "fixed_vs_ml_options_head_to_head.csv", index=False)
        paired.to_csv(out / "fixed_vs_ml_options_trade_pairs.csv", index=False)
        if len(by_month):
            by_month.to_csv(out / "fixed_vs_ml_options_by_month.csv", index=False)
        if len(by_side):
            by_side.to_csv(out / "fixed_vs_ml_options_by_side.csv", index=False)
        if len(choice_df):
            choice_df.to_csv(out / "fixed_vs_ml_options_contract_choice.csv", index=False)
        print(f"Wrote comparison CSVs under {out}")

    return {
        "metrics": comparison.reset_index(),
        "side_by_side": side_by_side,
        "head_to_head": head_to_head,
        "paired_trades": paired,
        "by_month": by_month,
        "by_side": by_side,
        "choice_diff": choice_df,
        "fixed_picks": fixed_picks,
        "ml_picks": ml_picks,
        "headline": {
            "verdict": verdict,
            "mean_lift": mean_lift,
            "mean_lift_pct": mean_lift_pct,
            "mean_fixed": mean_f,
            "mean_ml": mean_m,
            "n_trades": n_trades,
        },
    }




TEMPORAL_PANEL_PATH = paths.option_artifact_dir / "option_candidate_panel_temporal_is_oos.parquet"
TEMPORAL_SPLIT_SUMMARY_PATH = paths.option_artifact_dir / "option_ranker_temporal_split_summary.json"
EQUITY_SCORE_PATH = META_DIR / "equity_family_scores_on_option_entry_dates.parquet"
OPTION_STACK_PATH = META_DIR / "option_rows_with_equity_scores.parquet"
META_MODEL_PATH = META_DIR / "meta_stack_ranker.pkl"
META_SUMMARY_PATH = META_DIR / "meta_stack_summary.json"
OOS_COMPARISON_PATH = META_DIR / "option_ranker_oos_vs_baseline.csv"
EQUITY_MODEL_RESULTS_PATH = META_DIR / "equity_family_model_results.csv"


def prepare_temporal_option_panel(
    panel_path: Path,
    *,
    train_end: pd.Timestamp,
    eval_start: pd.Timestamp,
    output_path: Path,
    train_top_k_by_return: int = 100,
) -> dict:
    raw = pd.read_parquet(panel_path)
    if raw.empty:
        raise RuntimeError(f"Option panel empty: {panel_path}")
    work = raw.copy()
    work["entry_date"] = pd.to_datetime(work["entry_date"], errors="coerce").dt.normalize()
    if "option_return" in work.columns:
        work["option_return"] = pd.to_numeric(work["option_return"], errors="coerce")
    work["symbol"] = work["symbol"].astype(str).str.upper()
    if "rank_y" not in work.columns:
        if "option_return" not in work.columns:
            raise RuntimeError("Option panel requires unified rank_y or legacy option_return")
        work["rank_y"] = work.groupby("trade_id")["option_return"].rank(method="average", pct=True, ascending=True)
    work["rank_y"] = pd.to_numeric(work["rank_y"], errors="coerce")
    work = work.dropna(subset=["trade_id", "entry_date", "rank_y", "symbol"])
    work = _filter_oracle_entry_options(work, target_col="rank_y")
    if int(train_top_k_by_return) > 0:
        work = pd.concat(
            [_select_diverse_option_candidates(group, int(train_top_k_by_return)) for _, group in work.groupby("trade_id", sort=False)],
            ignore_index=True, sort=False,
        )

    trades = work.groupby("trade_id", as_index=False).agg(entry_date=("entry_date", "min"), symbol=("symbol", "first"))
    train_ids = set(trades.loc[trades["entry_date"].le(train_end), "trade_id"].astype(str))
    oos_ids = set(trades.loc[trades["entry_date"].ge(eval_start), "trade_id"].astype(str))
    if not train_ids or not oos_ids:
        raise RuntimeError(f"Empty split train={len(train_ids)} forward={len(oos_ids)}")

    train_rows = work.loc[work["trade_id"].astype(str).isin(train_ids)].copy()
    oos_rows = work.loc[work["trade_id"].astype(str).isin(oos_ids)].copy()
    train_before = len(train_rows)
    # rank_y is the immutable unified target; never recreate it from option_return here.

    filtered = pd.concat([train_rows, oos_rows], ignore_index=True, sort=False)
    max_is = pd.Timestamp(trades.loc[trades["trade_id"].astype(str).isin(train_ids), "entry_date"].max()).normalize()
    min_oos = pd.Timestamp(trades.loc[trades["trade_id"].astype(str).isin(oos_ids), "entry_date"].min()).normalize()
    if not (max_is < min_oos):
        raise AssertionError(f"entry overlap max_is={max_is.date()} min_oos={min_oos.date()}")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    filtered.to_parquet(output_path, index=False)
    return {
        "train_end": train_end.date().isoformat(),
        "eval_start": eval_start.date().isoformat(),
        "train_option_rows_before_top_k": int(train_before),
        "train_option_rows": int(len(train_rows)),
        "forward_option_rows": int(len(oos_rows)),
        "is_trades": int(len(train_ids)),
        "oos_trades": int(len(oos_ids)),
        "max_is_entry": max_is.date().isoformat(),
        "min_oos_entry": min_oos.date().isoformat(),
    }


def _ensure_fixed_near_atm_score(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    if "fixed_near_atm_score" in out.columns and pd.to_numeric(out["fixed_near_atm_score"], errors="coerce").notna().any():
        return out
    if "dte" in out.columns and "dte_gap" not in out.columns:
        out["dte_gap"] = (pd.to_numeric(out["dte"], errors="coerce") - float(OPTION_TARGET_DTE)).abs()
    terms = []
    for col in ("dte_gap", "abs_moneyness", "spread_pct"):
        if col not in out.columns:
            continue
        v = pd.to_numeric(out[col], errors="coerce")
        scale = v.abs().median()
        if pd.notna(scale) and float(scale) > 0:
            v = v / float(scale)
        terms.append(v.fillna(v.max() if v.notna().any() else 0.0))
    out["fixed_near_atm_score"] = -sum(terms) if terms else np.nan
    return out


def _wide_family_scores(strategy_scores: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    if strategy_scores is None or strategy_scores.empty:
        return pd.DataFrame(columns=["symbol", "date"]), []
    ss = strategy_scores.copy()
    ss["symbol"] = ss["symbol"].astype(str).str.upper()
    ss["date"] = pd.to_datetime(ss["date"], errors="coerce").dt.normalize()
    ss["strategy_source"] = ss["strategy_source"].astype(str)
    if "net_score" not in ss.columns and {"long_score", "short_score"}.issubset(ss.columns):
        ss["net_score"] = pd.to_numeric(ss["long_score"], errors="coerce") - pd.to_numeric(ss["short_score"], errors="coerce")
    frames = []
    feature_cols: list[str] = []
    for value_col in ("long_score",):
        if value_col not in ss.columns:
            continue
        ss[value_col] = pd.to_numeric(ss[value_col], errors="coerce")
        wide = ss.pivot_table(index=["symbol", "date"], columns="strategy_source", values=value_col, aggfunc="mean")
        wide = wide.rename(columns={c: f"{value_col}__{str(c).replace('.', '_')}" for c in wide.columns})
        feature_cols.extend([str(c) for c in wide.columns])
        frames.append(wide)
    out = pd.concat(frames, axis=1).reset_index()
    for c in feature_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce").astype("float32")
    return out, feature_cols


def _fit_option_meta_ranker(train: pd.DataFrame, *, features: list[str], target_col: str, random_seed: int):
    feats = [c for c in features if c in train.columns and pd.to_numeric(train[c], errors="coerce").notna().any()]
    y = pd.to_numeric(train[target_col], errors="coerce")
    valid = y.notna()
    if not feats or int(valid.sum()) == 0:
        return None, feats
    config = OptionMetaRankerConfig(
        option_panel=Path('.'), equity_score_store=Path('.'), output_dir=Path('.'),
        n_estimators=max(50, int(N_ESTIMATORS)), random_seed=int(random_seed),
        model_backend="rapids_random_forest",
    )
    model, medians = _fit_option_ranker(train.loc[valid, feats], y.loc[valid], config)
    return {"model": model, "medians": medians, "backend": config.model_backend}, feats


def _predict_meta(model, frame: pd.DataFrame, *, features: list[str]) -> np.ndarray:
    if model is None or not features or frame.empty:
        return np.full(len(frame), np.nan)
    return np.clip(_predict_option_ranker(
        model["model"], frame[features], medians=model["medians"], backend=model["backend"]
    ), 0.0, 1.0)


# --- guards ---
required = ["screened_equity_symbols", "feature_panel", "selected_feature_metadata", "oracle_label_rows"]
missing = [n for n in required if n not in globals()]
if missing:
    raise RuntimeError(
        "Run Universe Screen, Feature Engineering, and Oracle Trade Labels first. Missing: "
        + ", ".join(missing)
    )
if not RUN_OPTION_RANKER_TRAINING:
    print("RUN_OPTION_RANKER_TRAINING=False; skip.")
elif not OPTION_PANEL_PATH.exists():
    raise FileNotFoundError(f"Missing option panel {OPTION_PANEL_PATH}")
else:
    # 1) temporal option panel
    temporal_split_summary = prepare_temporal_option_panel(
        OPTION_PANEL_PATH,
        train_end=train_end_ts,
        eval_start=eval_start_ts,
        output_path=TEMPORAL_PANEL_PATH,
        train_top_k_by_return=TRAIN_TOP_K_BY_RETURN,
    )
    TEMPORAL_SPLIT_SUMMARY_PATH.write_text(json.dumps(temporal_split_summary, indent=2, default=str), encoding="utf-8")
    display(pd.DataFrame([temporal_split_summary]).T.rename(columns={0: "value"}))

    option_panel = pd.read_parquet(TEMPORAL_PANEL_PATH)
    option_panel = _ensure_fixed_near_atm_score(option_panel)
    option_panel["entry_date"] = pd.to_datetime(option_panel["entry_date"], errors="coerce").dt.normalize()
    option_panel["symbol"] = option_panel["symbol"].astype(str).str.upper()
    option_features = [
        c for c in OPTION_FEATURES
        if c in option_panel.columns and pd.to_numeric(option_panel[c], errors="coerce").notna().any()
    ]
    score_keys = (
        option_panel[["symbol", "entry_date"]]
        .drop_duplicates()
        .rename(columns={"entry_date": "date"})
        .reset_index(drop=True)
    )
    print({"option_rows": len(option_panel), "option_features": len(option_features), "entry_dates": len(score_keys)}, flush=True)

    # 2) Stage A — same as trading_app_v2 MoE family training on equity oracle labels
    if RUN_EQUITY_FAMILY_MODELS:
        equity_labels = oracle_label_rows.copy()
        equity_labels["symbol"] = equity_labels["symbol"].astype(str).str.upper()
        equity_labels["date"] = pd.to_datetime(equity_labels["date"], errors="coerce").dt.normalize()
        equity_labels = equity_labels.dropna(subset=["symbol", "date", "collapsed_label"])

        from dataclasses import fields as _dc_fields
        _equity_cfg_kwargs = {
            "experiment_name": "trading_app_v2_option_meta_equity_base",
            "mode": "classifier",
            "min_market_cap": MIN_MARKET_CAP,
            "symbols": tuple(screened_equity_symbols),
            "start_date": DATA_START,
            "end_date": optional_date(DATA_END),
            "train_end": option_train_end_value,
            "oos_start": option_eval_start_value,
            "score_start": DATA_START,
            "fit_all_available_data": False,
            "refresh_missing_fmp_data": False,
            "top_k_values": (1,),
            "strategy_sources": (
                tuple(
                    sorted(
                        (
                            selected_feature_metadata["source"].astype(str)
                            + "."
                            + selected_feature_metadata["family"].astype(str)
                        ).unique()
                    )
                )
                if ALL_FEATURE_FAMILIES
                else tuple(FEATURE_FAMILIES)
            ),
            "feature_nxt": ("raw",),
            "ae_config": None,
            "target_label_mode": "oracle_only",
            "oracle_frequencies": ("YE",),
            "oracle_k_min": 1,
            "oracle_k_max": max(ORACLE_YE_K),
            "quant_warehouse_root": None,
            "run_zipline_backtests": False,
            "include_yearly_vectorized_diagnostics": False,
            "backtesting_py_symbol_cases_per_side": 0,
            "run_model_diagnostics": False,
            "run_trading_diagnostics": False,
            "log_mlflow": False,
            "random_seed": RANDOM_SEED,
        }
        _allowed = {f.name for f in _dc_fields(MLTradingExperimentConfig)}
        equity_config = MLTradingExperimentConfig(**{k: v for k, v in _equity_cfg_kwargs.items() if k in _allowed})
        display(asdict(equity_config))
        print("[stage-a] _train_family_models on equity oracle labels...", flush=True)
        t0 = perf_counter()
        model_results, models = _train_family_models(
            equity_config,
            feature_panel,
            selected_feature_metadata,
            equity_labels,
            train_end=train_end_ts,
            oos_start=eval_start_ts,
            fit_all_available_data=False,
            run_model_diagnostics=False,
        )
        print(f"[stage-a] trained {len(models)} family models in {perf_counter() - t0:.1f}s", flush=True)
        display(model_results.head(50))
        model_results.to_csv(EQUITY_MODEL_RESULTS_PATH, index=False)

        # Score on feature panel (v2 path), then keep only option entry dates
        print("[stage-a] _score_family_models...", flush=True)
        t1 = perf_counter()
        strategy_scores, _ensemble = _score_family_models(
            equity_config,
            feature_panel,
            models,
            score_start=pd.Timestamp(DATA_START),
        )
        print(f"[stage-a] scored strategy_scores rows={len(strategy_scores)} in {perf_counter() - t1:.1f}s", flush=True)

        # Restrict to option entry dates (exact match, then asof fallback per symbol if needed)
        ss = strategy_scores.copy()
        ss["symbol"] = ss["symbol"].astype(str).str.upper()
        ss["date"] = pd.to_datetime(ss["date"], errors="coerce").dt.normalize()
        keys = score_keys.copy()
        keys["symbol"] = keys["symbol"].astype(str).str.upper()
        keys["date"] = pd.to_datetime(keys["date"], errors="coerce").dt.normalize()
        ss_on_entries = ss.merge(keys, on=["symbol", "date"], how="inner")
        if ss_on_entries.empty or ss_on_entries["strategy_source"].nunique() < max(1, len(models) // 2):
            print("[stage-a] exact entry-date score match sparse; using merge_asof per symbol", flush=True)
            parts = []
            for sym, g in keys.sort_values(["symbol", "date"]).groupby("symbol", sort=False):
                fam = ss.loc[ss["symbol"].eq(sym)].sort_values("date")
                if fam.empty:
                    continue
                # asof needs one row per date per strategy — do per strategy_source
                for src, fam_src in fam.groupby("strategy_source", sort=False):
                    m = pd.merge_asof(
                        g.sort_values("date"),
                        fam_src.sort_values("date"),
                        on="date",
                        by="symbol",
                        direction="backward",
                    )
                    parts.append(m)
            ss_on_entries = pd.concat(parts, ignore_index=True) if parts else ss_on_entries

        equity_wide, family_score_cols = _wide_family_scores(ss_on_entries)
        equity_wide.to_parquet(EQUITY_SCORE_PATH, index=False)
        print({"equity_wide_rows": len(equity_wide), "family_score_cols": len(family_score_cols)}, flush=True)
        del strategy_scores, ss, ss_on_entries
        gc.collect()
    else:
        if not EQUITY_SCORE_PATH.exists():
            raise FileNotFoundError("RUN_EQUITY_FAMILY_MODELS=False and no cached equity scores")
        equity_wide = pd.read_parquet(EQUITY_SCORE_PATH)
        family_score_cols = [c for c in equity_wide.columns if c not in {"symbol", "date"}]
        model_results = pd.DataFrame()
        models = {}

    if RUN_OPTION_META_STACK:
        # 3) Stage B — option meta: family scores + option features
        option_stack = option_panel.merge(
            equity_wide.rename(columns={"date": "entry_date"}),
            on=["symbol", "entry_date"],
            how="left",
        )
        # side-aware score per family (long→long_score, short→short_score)
        if "side" in option_stack.columns:
            for col in list(family_score_cols):
                if not str(col).startswith("long_score__"):
                    continue
                stem = str(col).removeprefix("long_score__")
                short_col = f"short_score__{stem}"
                side_col = f"side_score__{stem}"
                if short_col not in option_stack.columns:
                    continue
                side = option_stack["side"].astype(str).str.lower()
                option_stack[side_col] = np.where(
                    side.eq("long"),
                    pd.to_numeric(option_stack[col], errors="coerce"),
                    pd.to_numeric(option_stack[short_col], errors="coerce"),
                ).astype("float32")
                family_score_cols.append(side_col)
        family_score_cols = list(dict.fromkeys(family_score_cols))
        option_stack.to_parquet(OPTION_STACK_PATH, index=False)

        meta_features = [*option_features, *family_score_cols]
        meta_features = [c for c in meta_features if c in option_stack.columns]
        meta_train, meta_forward, _, _ = _training_and_scoring_frames(
            option_stack,
            train_end=option_train_end_value,
            eval_start=option_eval_start_value,
            train_on_all_data=False,
        )
        print(
            {
                "meta_train_rows": len(meta_train),
                "meta_forward_rows": len(meta_forward),
                "meta_features": len(meta_features),
                "n_option_features": len(option_features),
                "n_equity_score_features": len(family_score_cols),
            },
            flush=True,
        )

        selector_rows = []
        fixed = _evaluate_selector(
            meta_forward, score_col="fixed_near_atm_score", selector_name="fixed_near_atm", target_col=TARGET_COL
        )
        selector_rows.append({"model": "fixed_near_atm", **fixed})

        t2 = perf_counter()
        meta_model, meta_used = _fit_option_meta_ranker(
            meta_train, features=meta_features, target_col=TARGET_COL, random_seed=RANDOM_SEED + 999
        )
        meta_forward = meta_forward.copy()
        meta_forward["pred_meta_stack_rank"] = _predict_meta(meta_model, meta_forward, features=meta_used)
        meta_metrics = _evaluate_selector(
            meta_forward, score_col="pred_meta_stack_rank", selector_name="meta_stack", target_col=TARGET_COL
        )
        selector_rows.append({"model": "ml_options", **meta_metrics})
        print(
            f"[stage-b] ml_options (meta-stack) features={len(meta_used)} elapsed={perf_counter() - t2:.1f}s "
            f"fwd_mean_ret={meta_metrics.get('mean_option_return')} trades={meta_metrics.get('trades')}",
            flush=True,
        )

        if meta_model is not None:
            with META_MODEL_PATH.open("wb") as handle:
                pickle.dump(
                    {
                        "schema_version": 4,
                        "equity_score_contract": "family_long_probability_only",
                        "option_target_contract": OPTION_TARGET_CONTRACT,
                        "model_backend": meta_model["backend"],
                        "model": meta_model["model"],
                        "medians": meta_model["medians"],
                        "features": meta_used,
                        "option_features": option_features,
                        "equity_score_cols": family_score_cols,
                    },
                    handle,
                )

        # Store ml_options as the display name for the meta-stack ranker
        selector_rows = [
            {"model": "fixed_near_atm", "score_col": "fixed_near_atm_score", **fixed},
            {"model": "ml_options", "score_col": "pred_meta_stack_rank", **meta_metrics},
        ]
        selector_summary = pd.DataFrame(selector_rows)
        selector_summary.to_csv(META_DIR / "selector_summary.csv", index=False)

        comparison_bundle = display_fixed_vs_ml_options_comparison(
            meta_forward,
            fixed_score_col="fixed_near_atm_score",
            ml_score_col="pred_meta_stack_rank",
            fixed_name="fixed_near_atm",
            ml_name="ml_options",
            output_dir=META_DIR,
        )
        comparison_bundle["metrics"].to_csv(OOS_COMPARISON_PATH, index=False)

        META_SUMMARY_PATH.write_text(
            json.dumps(
                {
                    "design": "trading_app_v2 _train_family_models (equity) + option meta (family scores + option features)",
                    "train_end": option_train_end_value,
                    "eval_start": option_eval_start_value,
                    "min_market_cap": MIN_MARKET_CAP,
                    "equity_models": int(len(model_results)) if isinstance(model_results, pd.DataFrame) else None,
                    "option_features": option_features,
                    "equity_score_cols": family_score_cols,
                    "meta_features": meta_used,
                    "fixed_near_atm": fixed,
                    "ml_options": meta_metrics,
                    "headline": comparison_bundle.get("headline"),
                    "head_to_head": comparison_bundle["head_to_head"].to_dict(orient="records"),
                },
                indent=2,
                default=str,
            ),
            encoding="utf-8",
        )
        print(f"Wrote artifacts under {META_DIR}", flush=True)
    else:
        print("RUN_OPTION_META_STACK=False")


## Artifact Check

Read-only inspection of local ranking / label artifacts.

In [ ]:
artifact_candidates = [
    ORACLE_TRADES_PATH,
    ORACLE_TRADES_UNIQUE_PATH,
    OPTION_COVERAGE_PATH,
    ORACLE_TRADES_IN_COVERAGE_PATH,
    ORACLE_TRADES_BEFORE_COVERAGE_PATH,
    OPTION_PANEL_PATH,
    paths.option_artifact_dir / "option_candidate_panel_temporal_is_oos.parquet",
    paths.option_artifact_dir / "option_ranker_temporal_split_summary.json",
    paths.option_artifact_dir / "option_ranker_oos_vs_baseline.csv",
    paths.option_artifact_dir / "option_meta_stack" / "selector_summary.csv",
    paths.option_artifact_dir / "option_meta_stack" / "meta_stack_summary.json",
    paths.option_artifact_dir / "option_meta_stack" / "equity_family_scores_on_option_entry_dates.parquet",
    paths.option_artifact_dir / "option_meta_stack" / "option_rows_with_equity_scores.parquet",
    OPTION_LABEL_SUMMARY_PATH,
    *sorted(paths.option_artifact_dir.glob("*.csv")),
]
rows = []
for path in artifact_candidates:
    if path.exists():
        rows.append(
            {
                "path": str(path.relative_to(paths.artifact_root) if paths.artifact_root in path.parents else path),
                "bytes": path.stat().st_size,
                "mtime": pd.Timestamp(path.stat().st_mtime, unit="s"),
            }
        )
display(pd.DataFrame(rows))

for name in ("family_summary.csv", "selector_summary.csv", "basket_summary.csv"):
    summary_path = paths.option_artifact_dir / name
    if summary_path.exists():
        print(f"--- {name} ---")
        display(pd.read_csv(summary_path).head(20))
    else:
        print(f"Missing {summary_path} (train ranker to create)")
